# UK Medicine Trade: Trend Analysis, 2022–2026

HMRC Overseas Trade in Goods, HS chapter 30 (pharmaceutical products), January 2022 to July 2026. Official statistics under the Open Government Licence. The notebook reads the two HMRC files from a Google Drive folder called Project 3.

**Questions**

1. Is the UK selling more or fewer medicines abroad?
2. Who is buying?
3. Is it leaning more on imports, and who supplies them?

**Answers**

1. Less in value. Exports averaged £1,884m a month in 2022 and £1,749m in 2025.
2. The United States, Belgium and Germany, with 25.2%, 11.6% and 10.4% of export value over the whole period.
3. Since a near balance in 2023 the UK has bought more than it has sold, by a wider margin each year. Belgium, the United States, the Netherlands and Germany supply the most, with 13.7%, 12.9%, 12.6% and 12.0% of import value over the whole period.

Amounts are in pounds, not adjusted for inflation. 2026 holds January to July only. This is descriptive analysis: it shows what moved, not why.

In [ ]:
# Connected Google Drive, which holds the Project 3 folder with the HMRC files and the saved tables.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Process phase

### Step 1: load the files and make working copies

In [ ]:
# Loaded the two HMRC files from the Project 3 folder in Drive: country, with one row per country, flow and month, and
# product, with one row per product code, flow and month.
# Why: the original files are never changed. All the cleaning is done on copies, so any figure can be checked back
# against them.
import pandas as pd
country = pd.read_csv('/content/drive/MyDrive/Project 3/HMRC Medicines by Country (Jan 2022 - Jul 2026).csv')
product = pd.read_csv('/content/drive/MyDrive/Project 3/HMRC Medicines by Product (Jan 2022 - Jul 2026).csv')

In [ ]:
country.shape

(12776, 8)

In [ ]:
country.head()

,EU / NON EU,Continent,Country,Value (£),Net Mass (Kg),Flow Type,Year,Month
0,EU,European Union,Austria,6814787,48756.0,EU - Exports,2022,January
1,EU,European Union,Austria,11891585,74395.0,EU - Exports,2022,February
2,EU,European Union,Austria,7369406,65655.0,EU - Exports,2022,March
3,EU,European Union,Austria,7723548,64349.0,EU - Exports,2022,April
4,EU,European Union,Austria,14518099,17780.0,EU - Exports,2022,May


In [ ]:
country.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12776 entries, 0 to 12775
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   EU / NON EU    12776 non-null  object 
 1   Continent      12776 non-null  object 
 2   Country        12776 non-null  object 
 3   Value (£)      12776 non-null  int64  
 4   Net Mass (Kg)  12666 non-null  float64
 5   Flow Type      12776 non-null  object 
 6   Year           12776 non-null  int64  
 7   Month          12776 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 798.6+ KB


In [ ]:
product.shape

(10848, 10)

In [ ]:
product.head(3)

,Commodities,HS2,HS4,HS6,CN8,Value (£),Net Mass (Kg),Flow Type,Year,Month
0,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,2,NaN,EU - Exports,2022,January
1,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1350945,NaN,EU - Exports,2022,February
2,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1084760,NaN,EU - Exports,2022,March


In [ ]:
product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10848 entries, 0 to 10847
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Commodities    10848 non-null  object 
 1   HS2            10848 non-null  object 
 2   HS4            10848 non-null  object 
 3   HS6            10848 non-null  object 
 4   CN8            10848 non-null  object 
 5   Value (£)      10848 non-null  int64  
 6   Net Mass (Kg)  10738 non-null  float64
 7   Flow Type      10848 non-null  object 
 8   Year           10848 non-null  int64  
 9   Month          10848 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 847.6+ KB


In [ ]:
# Counted the rows of country that are exact repeats of an earlier row.
# Why: a repeated row would count the same trade twice.
# Found: none.
# Decision: nothing to fix.
country.duplicated().sum()

np.int64(0)

In [ ]:
# Counted the different values in each column of country.
# Why: this shows which columns hold a few labels and which hold many, and whether any column never changes.
# Found: no column holds a single value. Flow Type, Year and Month hold only a few values each.
country.nunique()

,0
EU / NON EU,2
Continent,9
Country,218
Value (£),12505
Net Mass (Kg),9433
Flow Type,4
Year,5
Month,12


In [ ]:
# Counted the rows of product that are exact repeats of an earlier row, as for country.
# Found: none.
# Decision: nothing to fix.
product.duplicated().sum()

np.int64(0)

In [ ]:
# Counted the different values in each column of product.
# Why: a column that holds the same value in every row tells nothing about any row.
# Found: Commodities and HS2 each hold a single value.
# Decision: drop both columns in the cleaning, further down.
product.nunique()

,0
Commodities,1
HS2,1
HS4,8
HS6,47
CN8,58
Value (£),10746
Net Mass (Kg),7949
Flow Type,4
Year,5
Month,12


In [ ]:
# Listed the different continents in country.
# Why: to see the labels HMRC uses, and whether any of them is not a place.
# Found: one label, Stores and Provisions, is not a place. The rest are regions.
country['Continent'].unique()

array(['European Union', 'Asia and Oceania', 'Eastern Europe exc EU',
       'Latin America and Caribbean', 'Middle East and N Africa',
       'North America', 'Stores and Provisions', 'Sub-Saharan Africa',
       'Western Europe exc EU'], dtype=object)

In [ ]:
# Listed the different countries in country.
# Why: to see the labels HMRC uses, and whether any of them is not a country.
# Found: three entries are not trading partners: Estimates, Stores & Provis. and UK. They are looked at one by one below.
country['Country'].unique()

array(['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia',
       'Denmark', 'Estimates', 'Estonia', 'Finland', 'France', 'Germany',
       'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania',
       'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Portugal',
       'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Stores & Provis.',
       'Sweden', 'UK', 'Afghanistan', 'American Samoa', 'Antarctica',
       'Australia', 'Bangladesh', 'Bhutan', 'Brunei', 'Cambodia', 'China',
       'Christmas Is', 'Cook Islands', 'Fiji', 'French Polynesia',
       'Hong Kong', 'India', 'Indonesia', 'Japan', 'Kiribati', 'Laos',
       'Macao', 'Malaysia', 'Maldives', 'Marshall Islands', 'Mongolia',
       'Myanmar', 'Nepal', 'New Zealand', 'Norfolk Island', 'Pakistan',
       'Palau', 'Papua New Guinea', 'Philippines', 'Samoa', 'Singapore',
       'Solomon Islands', 'South Georgia Is', 'South Korea', 'Sri Lanka',
       'Taiwan', 'Thailand', 'Timor-Leste', 'Tonga', 'Vanuatu', 'Vietn

In [ ]:
# Listed the different values in the HS4 column of product.
# Why: to see how the products are labelled.
# Found: each product label is a four-digit code followed by a very long description. Two labels are not products:
# Non Response Estimates and HS2 Below Threshold Trade. They are looked at below.
# Decision: short names for the product groups are added later in the cleaning.
product['HS4'].unique()

array(['30 🔸 Non Response Estimates', '30 HS2 Below Threshold Trade',
       '3001 Dried glands and other organs for organo-therapeutic uses, whether or not powdered; extracts of glands or other organs or their secretions, for organo-therapeutic uses; heparin and its salts; other human or animal substances prepared for therapeutic',
       '3002 Human blood; animal blood prepared for therapeutic, prophylactic or diagnostic uses; antisera and other blood fractions and immunological products, whether or not modified or obtained by means of biotechnological processes; vaccines, toxins, cultures',
       '3003 Medicaments consisting of two or more constituents mixed together for therapeutic or prophylactic uses, not in measured doses or put up for retail sale (excl. goods of heading 3002, 3005 or 3006)',
       '3004 Medicaments consisting of mixed or unmixed products for therapeutic or prophylactic uses, put up in measured doses "incl. those for transdermal administration" or in forms or 

In [ ]:
# Listed the rows of country where Net Mass (Kg) is blank.
# Why: value per kg needs a weight, so the rows without one have to be known before any analysis. The next two cells show
# the first half and the second half of these rows, because the table view hides the middle ones.
# Found: the rows shown are all Estimates rows, HMRC's own estimates, and none of them has a weight.
country[country['Net Mass (Kg)'].isna()]



,EU / NON EU,Continent,Country,Value (£),Net Mass (Kg),Flow Type,Year,Month
769,EU,European Union,Estimates,2,NaN,EU - Exports,2022,January
770,EU,European Union,Estimates,1350945,NaN,EU - Exports,2022,February
771,EU,European Union,Estimates,1084760,NaN,EU - Exports,2022,March
772,EU,European Union,Estimates,1369748,NaN,EU - Exports,2022,April
773,EU,European Union,Estimates,1396,NaN,EU - Exports,2022,May
...,...,...,...,...,...,...,...,...
874,EU,European Union,Estimates,294267,NaN,EU - Imports,2026,March
875,EU,European Union,Estimates,19543,NaN,EU - Imports,2026,April
876,EU,European Union,Estimates,352835,NaN,EU - Imports,2026,May
877,EU,European Union,Estimates,13598229,NaN,EU - Imports,2026,June


In [ ]:
blanks = country[country['Net Mass (Kg)'].isna()]
blanks.head(55)


,EU / NON EU,Continent,Country,Value (£),Net Mass (Kg),Flow Type,Year,Month
769,EU,European Union,Estimates,2,NaN,EU - Exports,2022,January
770,EU,European Union,Estimates,1350945,NaN,EU - Exports,2022,February
771,EU,European Union,Estimates,1084760,NaN,EU - Exports,2022,March
772,EU,European Union,Estimates,1369748,NaN,EU - Exports,2022,April
773,EU,European Union,Estimates,1396,NaN,EU - Exports,2022,May
774,EU,European Union,Estimates,1339869,NaN,EU - Exports,2022,June
775,EU,European Union,Estimates,1369473,NaN,EU - Exports,2022,July
776,EU,European Union,Estimates,5839861,NaN,EU - Exports,2022,August
777,EU,European Union,Estimates,1187945,NaN,EU - Exports,2022,September
778,EU,European Union,Estimates,40472,NaN,EU - Exports,2022,October


In [ ]:
blanks.tail(55)

,EU / NON EU,Continent,Country,Value (£),Net Mass (Kg),Flow Type,Year,Month
824,EU,European Union,Estimates,0,NaN,EU - Imports,2022,January
825,EU,European Union,Estimates,0,NaN,EU - Imports,2022,February
826,EU,European Union,Estimates,0,NaN,EU - Imports,2022,March
827,EU,European Union,Estimates,9185,NaN,EU - Imports,2022,April
828,EU,European Union,Estimates,2324,NaN,EU - Imports,2022,May
829,EU,European Union,Estimates,890,NaN,EU - Imports,2022,June
830,EU,European Union,Estimates,1203,NaN,EU - Imports,2022,July
831,EU,European Union,Estimates,16461,NaN,EU - Imports,2022,August
832,EU,European Union,Estimates,9769,NaN,EU - Imports,2022,September
833,EU,European Union,Estimates,10899,NaN,EU - Imports,2022,October


In [ ]:
# Listed the rows of product where Net Mass (Kg) is blank, as for country.
# Found: the rows shown are all Non Response Estimates rows, and none of them has a weight.
product[product['Net Mass (Kg)'].isna()]

,Commodities,HS2,HS4,HS6,CN8,Value (£),Net Mass (Kg),Flow Type,Year,Month
0,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,2,NaN,EU - Exports,2022,January
1,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1350945,NaN,EU - Exports,2022,February
2,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1084760,NaN,EU - Exports,2022,March
3,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1369748,NaN,EU - Exports,2022,April
4,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1396,NaN,EU - Exports,2022,May
...,...,...,...,...,...,...,...,...,...,...
105,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,294267,NaN,EU - Imports,2026,March
106,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,19543,NaN,EU - Imports,2026,April
107,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,352835,NaN,EU - Imports,2026,May
108,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,13598229,NaN,EU - Imports,2026,June


In [ ]:
nulls = product[product['Net Mass (Kg)'].isna()]
nulls.head(55)


,Commodities,HS2,HS4,HS6,CN8,Value (£),Net Mass (Kg),Flow Type,Year,Month
0,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,2,NaN,EU - Exports,2022,January
1,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1350945,NaN,EU - Exports,2022,February
2,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1084760,NaN,EU - Exports,2022,March
3,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1369748,NaN,EU - Exports,2022,April
4,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1396,NaN,EU - Exports,2022,May
5,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1339869,NaN,EU - Exports,2022,June
6,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1369473,NaN,EU - Exports,2022,July
7,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,5839861,NaN,EU - Exports,2022,August
8,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1187945,NaN,EU - Exports,2022,September
9,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,40472,NaN,EU - Exports,2022,October


In [ ]:
nulls.tail(55)

,Commodities,HS2,HS4,HS6,CN8,Value (£),Net Mass (Kg),Flow Type,Year,Month
55,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2022,January
56,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2022,February
57,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2022,March
58,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,9185,NaN,EU - Imports,2022,April
59,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,2324,NaN,EU - Imports,2022,May
60,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,890,NaN,EU - Imports,2022,June
61,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1203,NaN,EU - Imports,2022,July
62,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,16461,NaN,EU - Imports,2022,August
63,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,9769,NaN,EU - Imports,2022,September
64,All Commodities,30 Pharmaceutical products,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,10899,NaN,EU - Imports,2022,October


In [ ]:
# Listed the rows of country where Country is UK.
# Why: the UK is not a trading partner of itself, so a UK row needs explaining before it goes into any list of partners.
# Found: the only UK row has £0 and 0 kg. Its labels contradict each other: the EU status says EU, but the flow is
# Non EU - Exports.
# Decision: drop the row in the cleaning. It holds no money, so no total changes.
country[country['Country']=='UK']

,EU / NON EU,Continent,Country,Value (£),Net Mass (Kg),Flow Type,Year,Month
3068,EU,European Union,UK,0,0.0,Non EU - Exports,2025,July


In [ ]:
# Listed the rows of country where Country is Stores & Provis.
# Why: this is not a country, so it must stay out of any list of trading partners.
# Found: the rows are small, and they appear under both the EU and the non-EU labels.
# Decision: label these rows instead of deleting them, so the totals still match HMRC's.
country[country['Country']=='Stores & Provis.']

,EU / NON EU,Continent,Country,Value (£),Net Mass (Kg),Flow Type,Year,Month
2956,EU,European Union,Stores & Provis.,8081,126.0,EU - Exports,2025,August
2957,EU,European Union,Stores & Provis.,1755,102.0,EU - Exports,2026,April
10462,NON-EU,Stores and Provisions,Stores & Provis.,1146,10.0,Non EU - Exports,2022,March
10463,NON-EU,Stores and Provisions,Stores & Provis.,23408,272.0,Non EU - Exports,2022,May
10464,NON-EU,Stores and Provisions,Stores & Provis.,1313,6.0,Non EU - Exports,2022,December
10465,NON-EU,Stores and Provisions,Stores & Provis.,3554,400.0,Non EU - Exports,2023,February
10466,NON-EU,Stores and Provisions,Stores & Provis.,918,126.0,Non EU - Exports,2023,July
10467,NON-EU,Stores and Provisions,Stores & Provis.,1339,0.0,Non EU - Exports,2023,October
10468,NON-EU,Stores and Provisions,Stores & Provis.,1474,280.0,Non EU - Exports,2024,January
10469,NON-EU,Stores and Provisions,Stores & Provis.,15624,98.0,Non EU - Exports,2024,April


In [ ]:
# Listed the rows of country where Country is Estimates.
# Why: this is not a country either, and these are the rows that hold the blank weights seen above.
# Found: HMRC's own estimate rows, in the EU flows only, and none of them has a weight.
# Decision: label these rows instead of deleting them, because they hold real money.
country[country['Country']=='Estimates']

,EU / NON EU,Continent,Country,Value (£),Net Mass (Kg),Flow Type,Year,Month
769,EU,European Union,Estimates,2,NaN,EU - Exports,2022,January
770,EU,European Union,Estimates,1350945,NaN,EU - Exports,2022,February
771,EU,European Union,Estimates,1084760,NaN,EU - Exports,2022,March
772,EU,European Union,Estimates,1369748,NaN,EU - Exports,2022,April
773,EU,European Union,Estimates,1396,NaN,EU - Exports,2022,May
...,...,...,...,...,...,...,...,...
874,EU,European Union,Estimates,294267,NaN,EU - Imports,2026,March
875,EU,European Union,Estimates,19543,NaN,EU - Imports,2026,April
876,EU,European Union,Estimates,352835,NaN,EU - Imports,2026,May
877,EU,European Union,Estimates,13598229,NaN,EU - Imports,2026,June


In [ ]:
# Listed the rows of product where HS4 is 30 HS2 Below Threshold Trade.
# Why: this is not a product, so it must stay out of any table by product.
# Found: the rows have no product detail, because the same label sits in the HS4, HS6 and CN8 columns. They are in the
# EU flows only.
# Decision: label these rows instead of deleting them, because they hold real money.
product[product['HS4']=='30 HS2 Below Threshold Trade']

,Commodities,HS2,HS4,HS6,CN8,Value (£),Net Mass (Kg),Flow Type,Year,Month
110,All Commodities,30 Pharmaceutical products,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,325386,9322.0,EU - Exports,2022,January
111,All Commodities,30 Pharmaceutical products,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,435358,3894.0,EU - Exports,2022,February
112,All Commodities,30 Pharmaceutical products,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,479155,1957.0,EU - Exports,2022,March
113,All Commodities,30 Pharmaceutical products,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,1213536,20820.0,EU - Exports,2022,April
114,All Commodities,30 Pharmaceutical products,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,186523,23396.0,EU - Exports,2022,May
...,...,...,...,...,...,...,...,...,...,...
213,All Commodities,30 Pharmaceutical products,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,986448,1776.0,EU - Imports,2026,February
214,All Commodities,30 Pharmaceutical products,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,627904,1012.0,EU - Imports,2026,March
215,All Commodities,30 Pharmaceutical products,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,751136,3068.0,EU - Imports,2026,April
216,All Commodities,30 Pharmaceutical products,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,30 HS2 Below Threshold Trade,833136,5872.0,EU - Imports,2026,May


In [ ]:
country.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12776 entries, 0 to 12775
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   EU / NON EU    12776 non-null  object 
 1   Continent      12776 non-null  object 
 2   Country        12776 non-null  object 
 3   Value (£)      12776 non-null  int64  
 4   Net Mass (Kg)  12666 non-null  float64
 5   Flow Type      12776 non-null  object 
 6   Year           12776 non-null  int64  
 7   Month          12776 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 798.6+ KB


In [ ]:
country['Continent'].str.strip()

,Continent
0,European Union
1,European Union
2,European Union
3,European Union
4,European Union
...,...
12771,Western Europe exc EU
12772,Western Europe exc EU
12773,Western Europe exc EU
12774,Western Europe exc EU


In [ ]:
# Compared each label in Continent with the same label trimmed of spaces at the start and end.
# Why: a stray space would make two labels that look the same count as different values.
# This is a trial on one column. The next cell counts the differences, and the cell after repeats the test on every text
# column of country.
country['Continent']!=country['Continent'].str.strip()

,Continent
0,False
1,False
2,False
3,False
4,False
...,...
12771,False
12772,False
12773,False
12774,False


In [ ]:
(country['Continent']!=country['Continent'].str.strip()).sum()

np.int64(0)

In [ ]:
# Counted, for each text column of country, the labels that change when the spaces at the start and end are trimmed.
# Why: a stray space would split one label into two in a count or a chart.
# Found: none in any column.
# Decision: nothing to fix.
for output in ['Continent', 'EU / NON EU', 'Country', 'Flow Type', 'Month']:
  print(output, (country[output]!=country[output].str.strip()).sum())

Continent 0
EU / NON EU 0
Country 0
Flow Type 0
Month 0


In [ ]:
product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10848 entries, 0 to 10847
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Commodities    10848 non-null  object 
 1   HS2            10848 non-null  object 
 2   HS4            10848 non-null  object 
 3   HS6            10848 non-null  object 
 4   CN8            10848 non-null  object 
 5   Value (£)      10848 non-null  int64  
 6   Net Mass (Kg)  10738 non-null  float64
 7   Flow Type      10848 non-null  object 
 8   Year           10848 non-null  int64  
 9   Month          10848 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 847.6+ KB


In [ ]:
# Counted the same for each text column of product, as for country.
# Found: none in any column.
# Decision: nothing to fix.
for result in ['Commodities', 'HS2', 'HS4', 'HS6', 'CN8', 'Flow Type', 'Month']:
  print(result, (product[result]!=product[result].str.strip()).sum())

Commodities 0
HS2 0
HS4 0
HS6 0
CN8 0
Flow Type 0
Month 0


In [ ]:
# Counted, for each text column of country, the labels with a double space inside.
# Why: the first test only sees spaces at the ends. A double space in the middle would also split one label into two.
# Found: none in any column.
# Decision: nothing to fix.
for output in ['Continent', 'EU / NON EU', 'Country', 'Flow Type', 'Month']:
  print(output, country[output].str.contains('  ').sum())

Continent 0
EU / NON EU 0
Country 0
Flow Type 0
Month 0


In [ ]:
# Counted the same for each text column of product, as for country.
# Found: none in any column.
# Decision: nothing to fix.
for result in ['Commodities', 'HS2', 'HS4', 'HS6', 'CN8', 'Flow Type', 'Month']:
  print(result, product[result].str.contains('  ').sum())

Commodities 0
HS2 0
HS4 0
HS6 0
CN8 0
Flow Type 0
Month 0


In [ ]:
# Made working copies of the two tables, as country_clean and product_clean.
# Why: .copy() makes a second, separate table, so nothing done to the copies can touch country and product.
country_clean = country.copy()
product_clean = product.copy()

In [ ]:
# Showed the shape of both original tables and of both working copies.
# Why: to check that each copy holds every row and column of its original.
# Found: each copy has the same shape as its original.
# Decision: nothing to fix.
print(country.shape)
print(product.shape)

print(country_clean.shape)
print(product_clean.shape)

(12776, 8)
(10848, 10)
(12776, 8)
(10848, 10)


In [ ]:
country_clean.info()
product_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12776 entries, 0 to 12775
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   EU / NON EU    12776 non-null  object 
 1   Continent      12776 non-null  object 
 2   Country        12776 non-null  object 
 3   Value (£)      12776 non-null  int64  
 4   Net Mass (Kg)  12666 non-null  float64
 5   Flow Type      12776 non-null  object 
 6   Year           12776 non-null  int64  
 7   Month          12776 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 798.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10848 entries, 0 to 10847
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Commodities    10848 non-null  object 
 1   HS2            10848 non-null  object 
 2   HS4            10848 non-null  object 
 3   HS6            10848 non-null  object 
 4   CN8       

### Step 2: tidy the column names

In [ ]:
# Renamed the columns of both clean tables to short lowercase names, for example Value (£) to value_gbp and
# Net Mass (Kg) to net_mass_kg.
# Why: HMRC's names have capitals, spaces and brackets, which are slow to type and easy to get wrong.
country_clean = country_clean.rename(columns={'EU / NON EU': 'eu_status', 'Continent': 'continent', 'Country': 'country', 'Value (£)': 'value_gbp', 'Net Mass (Kg)': 'net_mass_kg', 'Flow Type': 'flow_type', 'Year': 'year', 'Month': 'month'})
product_clean = product_clean.rename(columns={'HS4': 'hs4', 'HS6': 'hs6', 'CN8': 'cn8', 'Value (£)': 'value_gbp', 'Net Mass (Kg)': 'net_mass_kg', 'Flow Type': 'flow_type', 'Year': 'year', 'Month': 'month'})

### Step 3: build one real date

In [ ]:
# Joined month and year into text for the first rows of product_clean.
# Why: month names sort alphabetically, so on their own they cannot draw a timeline. This is the first step of building a
# real date. The next cells test the join on country_clean, then turn the text into dates, before any column is added.
(product_clean['month'] + ' ' + product_clean['year'].astype(str)).head()

,0
0,January 2022
1,February 2022
2,March 2022
3,April 2022
4,May 2022


In [ ]:
# Joined month and year into text for the first rows of country_clean, as for product_clean.
# Found: the text reads like January 2022.
(country_clean['month'] + ' ' + country_clean['year'].astype(str)).head()

,0
0,January 2022
1,February 2022
2,March 2022
3,April 2022
4,May 2022


In [ ]:
# Turned the joined text of product_clean into dates, and showed the first rows.
# Why: the format %B %Y tells pandas to read a full month name and a four-digit year. This tests it before a column is added.
# Found: the first rows come out as the first day of each month, stored as datetime64[ns].
pd.to_datetime(product_clean['month'] + ' ' + product_clean['year'].astype(str), format = '%B %Y').head()

,0
0,2022-01-01
1,2022-02-01
2,2022-03-01
3,2022-04-01
4,2022-05-01


In [ ]:
# Turned the joined text of country_clean into dates, as for product_clean.
# Found: the same result, the first day of each month, stored as datetime64[ns].
pd.to_datetime(country_clean['month'] + ' ' + country_clean['year'].astype(str), format = '%B %Y').head()

,0
0,2022-01-01
1,2022-02-01
2,2022-03-01
3,2022-04-01
4,2022-05-01


In [ ]:
# Added date to both clean tables: month and year joined into text, then read as a date.
# Why: the question is about change over time. A real date sorts in time order and can draw a timeline, which a month name
# cannot.
product_clean['date'] = pd.to_datetime(product_clean['month'] + ' ' + product_clean['year'].astype(str), format = '%B %Y')
country_clean['date'] = pd.to_datetime(country_clean['month'] + ' ' + country_clean['year'].astype(str), format = '%B %Y')

In [ ]:
# Showed the column types of both clean tables.
# Why: to check that date was added and is stored as a date, not as text.
# Found: date is datetime64[ns] in both tables, and none of its values is missing.
product_clean.info()
country_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10848 entries, 0 to 10847
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Commodities  10848 non-null  object        
 1   HS2          10848 non-null  object        
 2   hs4          10848 non-null  object        
 3   hs6          10848 non-null  object        
 4   cn8          10848 non-null  object        
 5   value_gbp    10848 non-null  int64         
 6   net_mass_kg  10738 non-null  float64       
 7   flow_type    10848 non-null  object        
 8   year         10848 non-null  int64         
 9   month        10848 non-null  object        
 10  date         10848 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(2), object(7)
memory usage: 932.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12776 entries, 0 to 12775
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------   

In [ ]:
# Counted the different dates in each clean table, and showed the earliest and the latest.
# Why: the period should run from January 2022 to July 2026, one date for each month.
# Found: both tables hold the same number of dates, from January 2022 to July 2026.
# Decision: nothing to fix.
print('country_clean', country_clean['date'].nunique(), country_clean['date'].min(), country_clean['date'].max())
print('product_clean', product_clean['date'].nunique(), product_clean['date'].min(), product_clean['date'].max())

country_clean 55 2022-01-01 00:00:00 2026-07-01 00:00:00
product_clean 55 2022-01-01 00:00:00 2026-07-01 00:00:00


In [ ]:
country_clean.shape

(12776, 9)

### Step 4: drop the ‘UK’ row

In [ ]:
# Dropped the UK row from country_clean, by keeping every row where country is not UK.
# Why: the row holds £0 and 0 kg and its labels contradict each other, so it adds nothing and does not belong in a list of
# trading partners.
country_clean = country_clean[country_clean['country']!= 'UK']

In [ ]:
# Showed the shape of country_clean after the drop.
# Why: to check that exactly one row was removed.
# Found: one row fewer than before, and the same number of columns.
country_clean.shape

(12775, 9)

In [ ]:
# Compared the value_gbp total of country_clean with the Value (£) total of country, the untouched original.
# Why: dropping the UK row should not change any money.
# Found: the two totals are equal.
# Decision: nothing to fix.
print('country_clean', country_clean['value_gbp'].sum())
print('country', country['Value (£)'].sum())

country_clean 209718700270
country 209718700270


### Step 5: label the odd rows instead of deleting them

In [ ]:
# Counted the rows of country_clean where country is Estimates.
# Why: these rows are about to be labelled, so it helps to know how many there are.
# Found: the same number as the blank weights seen earlier in country.info().
(country_clean['country']=='Estimates').sum()

np.int64(110)

In [ ]:
# Counted the rows of country_clean where country is Stores & Provis., as for Estimates.
# Why: to know how many rows there are to label.
(country_clean['country']=='Stores & Provis.').sum()

np.int64(22)

In [ ]:
# Added row_type to country_clean, with every row starting as partner.
# Why: the odd rows can then be relabelled instead of deleted. That keeps the totals matching HMRC's, and lets the
# country breakdowns pick the partner rows later.
country_clean['row_type']='partner'

In [ ]:
# Showed the shape of country_clean after adding row_type.
# Why: to check that one column was added and no row was lost.
# Found: one more column than before, and the same number of rows.
country_clean.shape

(12775, 10)

In [ ]:
# Set row_type to estimate for the rows of country_clean where country is Estimates.
# Why: these rows are HMRC's own estimates, not a country. Labelling them keeps their money in the totals and keeps them
# out of the country breakdowns.
country_clean.loc[country_clean['country'] == 'Estimates', 'row_type'] = 'estimate'


In [ ]:
# Counted the rows of each row_type in country_clean.
# Why: to check that the label landed on the Estimates rows and nowhere else.
# Found: the estimate count equals the number of Estimates rows counted earlier, and every other row is still partner.
country_clean['row_type'].value_counts()

,count
row_type,
partner,12665
estimate,110


In [ ]:
# Set row_type to stores for the rows of country_clean where country is Stores & Provis.
# Why: this is not a country either, so the rows are labelled instead of deleted, as for Estimates.
country_clean.loc[country_clean['country'] == 'Stores & Provis.', 'row_type'] = 'stores'

In [ ]:
# Counted the rows of each row_type in country_clean again.
# Why: to check that the stores label landed on the right rows and the estimate rows were not touched.
# Found: the three counts add up to every row of country_clean, and the estimate count is unchanged.
# Decision: nothing to fix.
country_clean['row_type'].value_counts()

,count
row_type,
partner,12643
estimate,110
stores,22


In [ ]:
# Counted the rows for each HS4 label in product_clean.
# Why: to see how many rows each label covers before the odd labels are relabelled.
# Found: each product code covers a thousand rows or more, and the two odd labels, Non Response Estimates and
# HS2 Below Threshold Trade, cover only about a hundred each.
product_clean['hs4'].value_counts()

,count
hs4,
"3002 Human blood; animal blood prepared for therapeutic, prophylactic or diagnostic uses; antisera and other blood fractions and immunological products, whether or not modified or obtained by means of biotechnological processes; vaccines, toxins, cultures",2649
"3004 Medicaments consisting of mixed or unmixed products for therapeutic or prophylactic uses, put up in measured doses ""incl. those for transdermal administration"" or in forms or packings for retail sale (excl. goods of heading 3002, 3005 or 3006)",2328
3006 Pharmaceutical preparations and products of subheadings 3006.10.10 to 3006.93.00,2327
"3003 Medicaments consisting of two or more constituents mixed together for therapeutic or prophylactic uses, not in measured doses or put up for retail sale (excl. goods of heading 3002, 3005 or 3006)",1138
"3005 Wadding, gauze, bandages and the like, e.g. dressings, adhesive plasters, poultices, impregnated or covered with pharmaceutical substances or put up for retail sale for medical, surgical, dental or veterinary purposes",1100
"3001 Dried glands and other organs for organo-therapeutic uses, whether or not powdered; extracts of glands or other organs or their secretions, for organo-therapeutic uses; heparin and its salts; other human or animal substances prepared for therapeutic",1088
30 🔸 Non Response Estimates,110
30 HS2 Below Threshold Trade,108


In [ ]:
# Added row_type to product_clean, with every row starting as product.
# Why: the odd rows can then be relabelled instead of deleted. That keeps the totals matching HMRC's, and lets the
# product breakdowns pick the product rows later.
product_clean['row_type'] = 'product'

In [ ]:
# Showed the shape of product_clean after adding row_type.
# Why: to check that one column was added and no row was lost.
# Found: one more column than before, and the same number of rows.
product_clean.shape

(10848, 12)

In [ ]:
# Set row_type to estimate for the rows of product_clean where hs4 is Non Response Estimates.
# Why: these rows are HMRC's own estimates, not a product. Labelling them keeps their money in the totals and keeps them
# out of the product breakdowns.
product_clean.loc[product_clean['hs4'] == '30 🔸 Non Response Estimates', 'row_type'] = 'estimate'

In [ ]:
# Set row_type to below_threshold for the rows of product_clean where hs4 is HS2 Below Threshold Trade.
# Why: these rows have no product detail, so they are not a product. They are labelled instead of deleted, because they
# hold real money.
product_clean.loc[product_clean['hs4'] == '30 HS2 Below Threshold Trade', 'row_type'] = 'below_threshold'

In [ ]:
# Counted the rows of each row_type in product_clean.
# Why: to check that the two labels landed on the odd rows and nowhere else.
# Found: the estimate and below_threshold counts equal the counts of those two labels seen above, and every other row is
# still a product.
# Decision: nothing to fix.
product_clean['row_type'].value_counts()

,count
row_type,
product,10630
estimate,110
below_threshold,108


### Step 6: drop the two constant columns

In [ ]:
# Counted the different values in Commodities in product_clean.
# Why: to confirm on the clean table that the column holds one value only, before it is dropped.
# Found: one value, as in the raw table.
product_clean['Commodities'].nunique()

1

In [ ]:
# Counted the different values in HS2 in product_clean, as for Commodities.
# Why: the same check, because HS2 also held one value in the raw table.
# Found: one value, as in the raw table.
product_clean['HS2'].nunique()

1

In [ ]:
# Dropped the columns HS2 and Commodities from product_clean.
# Why: each holds the same value in every row, so it tells nothing about any row.
product_clean = product_clean.drop(columns = ['HS2', 'Commodities'])

In [ ]:
# Listed the column names of product_clean.
# Why: to check that HS2 and Commodities are gone and nothing else was removed.
# Found: both are gone, and the other columns, including date and row_type, are still there.
product_clean.columns

Index(['hs4', 'hs6', 'cn8', 'value_gbp', 'net_mass_kg', 'flow_type', 'year',
       'month', 'date', 'row_type'],
      dtype='object')

### Step 7: name the product groups

In [ ]:
# Listed the different values in eu_status in country_clean.
# Why: to check that each value has one spelling, because a group spelled two ways would be counted as two groups.
# Found: EU and NON-EU only, each spelled one way.
country_clean['eu_status'].unique()

array(['EU', 'NON-EU'], dtype=object)

In [ ]:
# Listed the different values in flow_type in country_clean.
# Why: the same spelling check, and it shows the four flows that the whole analysis is built on.
# Found: EU and Non EU, each with Exports and Imports, each spelled one way. Non EU is worded differently here than in
# eu_status, where it is NON-EU.
# Decision: left as HMRC wrote them, so the code has to use the spelling of each column.
country_clean['flow_type'].unique()

array(['EU - Exports', 'EU - Imports', 'Non EU - Exports',
       'Non EU - Imports'], dtype=object)

In [ ]:
# Listed the different values in flow_type in product_clean, as for country_clean.
# Why: to check that both tables name the flows the same way.
# Found: the same four values as in country_clean, each spelled one way.
# Decision: nothing to fix.
product_clean['flow_type'].unique()

array(['EU - Exports', 'EU - Imports', 'Non EU - Exports',
       'Non EU - Imports'], dtype=object)

In [ ]:
# Sorted the country column of country_clean A to Z and listed each name once.
# Why: to check how the names are written, and whether any country appears under two spellings.
# Found: some names are abbreviated or hold symbols (for example Antigua:Barbuda, Bosnia & Herz., Br Ind Oc Terr), but
# no country appears under two spellings. The list also holds Estimates and Stores & Provis., the two rows labelled above
# as not countries.
# Decision: left as HMRC wrote them. Revisit if the dashboard uses a map.
country_clean['country'].sort_values().unique()

array(['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra',
       'Angola', 'Anguilla', 'Antarctica', 'Antigua:Barbuda', 'Argentina',
       'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan',
       'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus',
       'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia',
       'Bosnia & Herz.', 'Botswana', 'Br Ind Oc Terr', 'Br Virgin Is',
       'Brazil', 'Brunei', 'Bulgaria', 'Burkina', 'Burundi', 'Cambodia',
       'Cameroon', 'Canada', 'Cape Verde', 'Cayman Islands',
       'Cent Afr Rep', 'Chad', 'Chile', 'China', 'Christmas Is',
       'Colombia', 'Congo (Dem. Rep)', 'Congo (Republic)', 'Cook Islands',
       'Costa Rica', 'Croatia', 'Cuba', 'Curacao', 'Cyprus', 'Czechia',
       'Denmark', 'Djibouti', 'Dominica', 'Dominican Rep', 'Ecuador',
       'Egypt', 'El Salvador', 'Equat Guinea', 'Eritrea', 'Estimates',
       'Estonia', 'Eswatini', 'Ethiopia', 'Falkland Islands',
       'Faroe Islands', 'Fiji', 

In [ ]:
# Listed the different values in hs4 in product_clean.
# Why: to check that each product label has one spelling, and to see what the labels look like.
# Found: six product labels, each a four-digit code followed by a long description, and the two odd labels, Non Response
# Estimates and HS2 Below Threshold Trade, which are labelled in row_type above.
# Decision: the long labels are replaced by short group names in the next cells.
product_clean['hs4'].unique()

array(['30 🔸 Non Response Estimates', '30 HS2 Below Threshold Trade',
       '3001 Dried glands and other organs for organo-therapeutic uses, whether or not powdered; extracts of glands or other organs or their secretions, for organo-therapeutic uses; heparin and its salts; other human or animal substances prepared for therapeutic',
       '3002 Human blood; animal blood prepared for therapeutic, prophylactic or diagnostic uses; antisera and other blood fractions and immunological products, whether or not modified or obtained by means of biotechnological processes; vaccines, toxins, cultures',
       '3003 Medicaments consisting of two or more constituents mixed together for therapeutic or prophylactic uses, not in measured doses or put up for retail sale (excl. goods of heading 3002, 3005 or 3006)',
       '3004 Medicaments consisting of mixed or unmixed products for therapeutic or prophylactic uses, put up in measured doses "incl. those for transdermal administration" or in forms or 

In [ ]:
# Built a dictionary, group_names, that gives each of the six four-digit product codes a short name.
# Why: each HS4 label is a very long description, which is too long to read on a chart. The short names are plain-language
# versions of the six headings.
group_names = {'3001': 'Glands, organ extracts, heparin', '3002': 'Blood products, antisera, vaccines', '3003': 'Bulk mixed medicines', '3004': 'Packaged medicines', '3005': 'Dressings and bandages', '3006': 'Other pharmaceutical products'}

In [ ]:
group_names

{'3001': 'Glands, organ extracts, heparin',
 '3002': 'Blood products, antisera, vaccines',
 '3003': 'Bulk mixed medicines',
 '3004': 'Packaged medicines',
 '3005': 'Dressings and bandages',
 '3006': 'Other pharmaceutical products'}

In [ ]:
# Took the first four characters of each hs4 label in product_clean and listed the different results.
# Why: to test the codes that group_names will be looked up by, before they are used on the table.
# Found: the six product codes, 3001 to 3006, and two odd results that come from the two odd labels.
# Decision: the six codes match the keys of group_names. The odd results match no key, so their rows get a blank group.
product_clean['hs4'].str[:4].unique()

array(['30 🔸', '30 H', '3001', '3002', '3003', '3004', '3005', '3006'],
      dtype=object)

In [ ]:
# Added product_group to product_clean: the first four characters of hs4, looked up in group_names.
# Why: the product tables can then group by six short names instead of six long labels.
product_clean['product_group'] = product_clean['hs4'].str[:4].map(group_names)

In [ ]:
# Counted the rows of each product_group in product_clean, including the blanks.
# Why: to check that every product row got a group, and to see which rows got none.
# Found: the six group counts equal the counts of the matching HS4 labels seen earlier. The blank group holds the same
# number of rows as the estimate and below_threshold rows together.
# Decision: nothing to fix. The blank rows are the odd rows, which are not products.
product_clean['product_group'].value_counts(dropna=False)

,count
product_group,
"Blood products, antisera, vaccines",2649
Packaged medicines,2328
Other pharmaceutical products,2327
Bulk mixed medicines,1138
Dressings and bandages,1100
"Glands, organ extracts, heparin",1088
NaN,218


### Sanity checks 1 to 4

In [ ]:
# Summarised the number columns and the date column of product_clean: count, mean, spread, minimum, quartiles, maximum.
# Why: to look for impossible values, such as negative money or weights, and for dates outside the period.
# Found: no negative values. The minimum is 0 for value_gbp and net_mass_kg, and net_mass_kg has fewer values than
# value_gbp, so some weights are blank. The mean is well above the median, so a few large rows pull it up. The dates run
# from January 2022 to July 2026.
# Decision: nothing to fix here. The zeros and the blank weights are looked at in the next cells.
product_clean.describe()

,value_gbp,net_mass_kg,year,date
count,1.084800e+04,1.073800e+04,10848.000000,10848
mean,1.933248e+07,1.911311e+05,2023.806416,2024-03-27 16:37:41.946902528
min,0.000000e+00,0.000000e+00,2022.000000,2022-01-01 00:00:00
25%,1.872510e+05,6.570000e+02,2023.000000,2023-02-01 00:00:00
50%,9.873340e+05,1.030500e+04,2024.000000,2024-04-01 00:00:00
75%,6.830466e+06,6.481750e+04,2025.000000,2025-06-01 00:00:00
max,1.085281e+09,1.757509e+07,2026.000000,2026-07-01 00:00:00
std,7.897657e+07,1.037263e+06,1.337748,NaN


In [ ]:
# Summarised the number columns and the date column of country_clean, as for product_clean.
# Why: the same checks, on the country table.
# Found: the same pattern as product_clean: no negative values, a minimum of 0, some blank weights, a mean well above the
# median, and dates from January 2022 to July 2026.
# Decision: nothing to fix here.
country_clean.describe()

,value_gbp,net_mass_kg,year,date
count,1.277500e+04,1.266500e+04,12775.000000,12775
mean,1.641634e+07,1.620502e+05,2023.819256,2024-04-01 15:12:28.180038912
min,0.000000e+00,0.000000e+00,2022.000000,2022-01-01 00:00:00
25%,5.942350e+04,9.140000e+02,2023.000000,2023-02-01 00:00:00
50%,5.385300e+05,8.608000e+03,2024.000000,2024-04-01 00:00:00
75%,4.366171e+06,6.643100e+04,2025.000000,2025-06-01 00:00:00
max,7.867968e+08,8.987254e+06,2026.000000,2026-07-01 00:00:00
std,5.575281e+07,4.855663e+05,1.337622,NaN


In [ ]:
# Showed the rows of country_clean where value_gbp is 0.
# Why: a minimum of 0 means some rows carry no money, and they might be mistakes.
# Found: two kinds of row: estimate rows (EU flows only, no weight) and partner rows with a recorded zero (0 kg as well).
# Decision: kept. A zero adds nothing to a total, and these rows are HMRC's own records.
country_clean[country_clean['value_gbp'] == 0]

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
784,EU,European Union,Estimates,0,NaN,EU - Exports,2023,April,2023-04-01,estimate
793,EU,European Union,Estimates,0,NaN,EU - Exports,2024,January,2024-01-01,estimate
824,EU,European Union,Estimates,0,NaN,EU - Imports,2022,January,2022-01-01,estimate
825,EU,European Union,Estimates,0,NaN,EU - Imports,2022,February,2022-02-01,estimate
826,EU,European Union,Estimates,0,NaN,EU - Imports,2022,March,2022-03-01,estimate
842,EU,European Union,Estimates,0,NaN,EU - Imports,2023,July,2023-07-01,estimate
843,EU,European Union,Estimates,0,NaN,EU - Imports,2023,August,2023-08-01,estimate
844,EU,European Union,Estimates,0,NaN,EU - Imports,2023,September,2023-09-01,estimate
845,EU,European Union,Estimates,0,NaN,EU - Imports,2023,October,2023-10-01,estimate
849,EU,European Union,Estimates,0,NaN,EU - Imports,2024,February,2024-02-01,estimate


In [ ]:
# Showed the rows of product_clean where value_gbp is 0, as for country_clean.
# Why: the same check, on the product table.
# Found: the same two kinds of row: estimate rows (EU flows only, no weight) and product rows with a recorded zero (0 kg
# as well).
# Decision: kept, for the same reason.
product_clean[product_clean['value_gbp'] == 0]

,hs4,hs6,cn8,value_gbp,net_mass_kg,flow_type,year,month,date,row_type,product_group
15,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Exports,2023,April,2023-04-01,estimate,NaN
24,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Exports,2024,January,2024-01-01,estimate,NaN
55,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2022,January,2022-01-01,estimate,NaN
56,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2022,February,2022-02-01,estimate,NaN
57,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2022,March,2022-03-01,estimate,NaN
73,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2023,July,2023-07-01,estimate,NaN
74,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2023,August,2023-08-01,estimate,NaN
75,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2023,September,2023-09-01,estimate,NaN
76,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2023,October,2023-10-01,estimate,NaN
80,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,0,NaN,EU - Imports,2024,February,2024-02-01,estimate,NaN


In [ ]:
# Showed the rows of product_clean where net_mass_kg is 0.
# Why: value per kg cannot be worked out for a weight of 0, so these rows need a look. The next cells count them by
# row type.
product_clean[product_clean['net_mass_kg'] == 0]

,hs4,hs6,cn8,value_gbp,net_mass_kg,flow_type,year,month,date,row_type,product_group
223,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,38331,0.0,EU - Exports,2022,June,2022-06-01,product,"Glands, organ extracts, heparin"
224,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,10631,0.0,EU - Exports,2022,July,2022-07-01,product,"Glands, organ extracts, heparin"
234,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,10107,0.0,EU - Exports,2023,May,2023-05-01,product,"Glands, organ extracts, heparin"
375,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,4497,0.0,Non EU - Exports,2026,February,2026-02-01,product,"Glands, organ extracts, heparin"
471,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012090 Extracts of glands or other organs or...,3450,0.0,EU - Exports,2024,December,2024-12-01,product,"Glands, organ extracts, heparin"
...,...,...,...,...,...,...,...,...,...,...,...
10556,3006 Pharmaceutical preparations and products ...,300692 Waste pharmaceuticals,30069200 Waste pharmaceuticals,2335,0.0,EU - Imports,2025,February,2025-02-01,product,Other pharmaceutical products
10559,3006 Pharmaceutical preparations and products ...,300692 Waste pharmaceuticals,30069200 Waste pharmaceuticals,10179,0.0,EU - Imports,2025,June,2025-06-01,product,Other pharmaceutical products
10603,3006 Pharmaceutical preparations and products ...,300692 Waste pharmaceuticals,30069200 Waste pharmaceuticals,1560,0.0,Non EU - Exports,2026,February,2026-02-01,product,Other pharmaceutical products
10624,3006 Pharmaceutical preparations and products ...,300692 Waste pharmaceuticals,30069200 Waste pharmaceuticals,1614,0.0,Non EU - Imports,2026,February,2026-02-01,product,Other pharmaceutical products


In [ ]:
# Counted the rows of country_clean where net_mass_kg is blank.
# Why: the summary above showed fewer weights than values, so this gives the size of the gap.
# Found: the same number as the Estimates rows counted earlier.
country_clean['net_mass_kg'].isna().sum()

np.int64(110)

In [ ]:
# Counted the rows of product_clean where net_mass_kg is blank, as for country_clean.
# Why: the same check, on the product table.
# Found: the same number as in country_clean, and the same as the number of estimate rows.
product_clean['net_mass_kg'].isna().sum()

np.int64(110)

In [ ]:
# Showed the rows of country_clean where net_mass_kg is blank.
# Why: to see what the rows without a weight look like. The table view shows only the first and last few rows, so the next
# cells count them by row type.
country_clean[country_clean['net_mass_kg'].isna()]

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
769,EU,European Union,Estimates,2,NaN,EU - Exports,2022,January,2022-01-01,estimate
770,EU,European Union,Estimates,1350945,NaN,EU - Exports,2022,February,2022-02-01,estimate
771,EU,European Union,Estimates,1084760,NaN,EU - Exports,2022,March,2022-03-01,estimate
772,EU,European Union,Estimates,1369748,NaN,EU - Exports,2022,April,2022-04-01,estimate
773,EU,European Union,Estimates,1396,NaN,EU - Exports,2022,May,2022-05-01,estimate
...,...,...,...,...,...,...,...,...,...,...
874,EU,European Union,Estimates,294267,NaN,EU - Imports,2026,March,2026-03-01,estimate
875,EU,European Union,Estimates,19543,NaN,EU - Imports,2026,April,2026-04-01,estimate
876,EU,European Union,Estimates,352835,NaN,EU - Imports,2026,May,2026-05-01,estimate
877,EU,European Union,Estimates,13598229,NaN,EU - Imports,2026,June,2026-06-01,estimate


In [ ]:
# Showed the rows of product_clean where net_mass_kg is blank, as for country_clean.
# Why: the same check, on the product table.
product_clean[product_clean['net_mass_kg'].isna()]

,hs4,hs6,cn8,value_gbp,net_mass_kg,flow_type,year,month,date,row_type,product_group
0,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,2,NaN,EU - Exports,2022,January,2022-01-01,estimate,NaN
1,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1350945,NaN,EU - Exports,2022,February,2022-02-01,estimate,NaN
2,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1084760,NaN,EU - Exports,2022,March,2022-03-01,estimate,NaN
3,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1369748,NaN,EU - Exports,2022,April,2022-04-01,estimate,NaN
4,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1396,NaN,EU - Exports,2022,May,2022-05-01,estimate,NaN
...,...,...,...,...,...,...,...,...,...,...,...
105,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,294267,NaN,EU - Imports,2026,March,2026-03-01,estimate,NaN
106,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,19543,NaN,EU - Imports,2026,April,2026-04-01,estimate,NaN
107,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,352835,NaN,EU - Imports,2026,May,2026-05-01,estimate,NaN
108,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,13598229,NaN,EU - Imports,2026,June,2026-06-01,estimate,NaN


In [ ]:
# Counted the rows of each row_type among the rows of product_clean where net_mass_kg is blank.
# Why: the table view above shows only the first and last few rows, so counting is the quick way to see what kind of rows
# have no weight.
# Found: every row with a blank weight is an estimate row (HMRC labels them Non Response Estimates).
# Decision: kept. They carry value, so they stay in the value totals. Value per kg will leave them out, because there is
# no weight to divide by.
product_clean[product_clean['net_mass_kg'].isna()]['row_type'].value_counts()

,count
row_type,
estimate,110


In [ ]:
# Counted the rows of each row_type among the rows of country_clean where net_mass_kg is blank, as for product_clean.
# Why: to see whether the result matches the product table.
# Found: the same as product_clean. Every row with a blank weight is an estimate row (HMRC labels them Estimates).
# Decision: kept, for the same reason.
country_clean[country_clean['net_mass_kg'].isna()]['row_type'].value_counts()

,count
row_type,
estimate,110


In [ ]:
# Counted the rows of each row_type among the rows of product_clean where net_mass_kg is 0.
# Why: the rows with a blank weight were all estimates, so this checks what kind of rows have a weight of exactly 0.
# Found: every row with a weight of 0 is a product row. None are estimate rows.
# Decision: kept. Any value they carry stays in the value totals. Value per kg will leave them out, because dividing by a
# weight of 0 gives no usable figure.
product_clean[product_clean['net_mass_kg'] == 0]['row_type'].value_counts()

,count
row_type,
product,75


In [ ]:
# Counted the rows of each row_type among the rows of country_clean where net_mass_kg is 0, as for product_clean.
# Why: the same check, on the country table.
# Found: partner rows, plus a stores row (HMRC's Stores & Provis.). None are estimate rows.
# Decision: kept, for the same reason.
country_clean[country_clean['net_mass_kg'] == 0]['row_type'].value_counts()

,count
row_type,
partner,107
stores,1


In [ ]:
# Counted the different dates in each flow_type of country_clean.
# Why: if a flow had no rows in some month, its monthly total would quietly dip.
# Found: every flow has as many months as the whole table, so no flow is missing a month. This checks each flow as a whole,
# not each country within it.
# Decision: nothing to fix.
country_clean.groupby('flow_type')['date'].nunique()

,date
flow_type,
EU - Exports,55
EU - Imports,55
Non EU - Exports,55
Non EU - Imports,55


In [ ]:
# Counted the different dates in each flow_type of product_clean, as for country_clean.
# Why: the same check, on the product table.
# Found: the same as country_clean. Every flow has as many months as the whole table. This checks each flow as a whole,
# not each product within it.
# Decision: nothing to fix.
product_clean.groupby('flow_type')['date'].nunique()

,date
flow_type,
EU - Exports,55
EU - Imports,55
Non EU - Exports,55
Non EU - Imports,55


In [ ]:
# Listed the different values in eu_status in country_clean again.
# Why: to have the exact spellings in view for the cross-check of eu_status against flow_type below.
country_clean['eu_status'].unique()

array(['EU', 'NON-EU'], dtype=object)

In [ ]:
# Listed the different values in flow_type in country_clean again, as for eu_status.
# Why: the same reason, so both sets of spellings are in view for the cross-check below.
country_clean['flow_type'].unique()

array(['EU - Exports', 'EU - Imports', 'Non EU - Exports',
       'Non EU - Imports'], dtype=object)

In [ ]:
# Counted the rows of each eu_status within each flow_type of country_clean.
# Why: flow_type and eu_status both say whether a row is EU or non-EU, so they should never contradict.
# Found: each flow has one eu_status only, and it matches the flow's name (EU flows are EU, Non EU flows are NON-EU).
# Decision: nothing to fix. Either column can be used to split EU from non-EU.
country_clean.groupby('flow_type')['eu_status'].value_counts()

,,count
flow_type,eu_status,
EU - Exports,EU,1542
EU - Imports,EU,1526
Non EU - Exports,NON-EU,7275
Non EU - Imports,NON-EU,2432


### Step 8: check that cleaning changed no money (check 5)

In [ ]:
# Added up the value_gbp column of country_clean.
# Why: check 5, totals. This and the next three totals should be identical, because cleaning must not change any money.
country_clean['value_gbp'].sum()

np.int64(209718700270)

In [ ]:
# Added up the value_gbp column of product_clean.
# Why: the second total, from the product table.
product_clean['value_gbp'].sum()

np.int64(209718700270)

In [ ]:
# Added up the Value (£) column of country, the untouched original.
# Why: the third total, taken before any cleaning, to compare with the clean tables.
country['Value (£)'].sum()

np.int64(209718700270)

In [ ]:
# Added up the Value (£) column of product, the untouched original.
# Why: the fourth total, taken before any cleaning.
# Found: all four totals are the same, so cleaning did not change any money, and the country and product files agree in total.
# Decision: nothing to fix.
product['Value (£)'].sum()

np.int64(209718700270)

In [ ]:
# Counted the rows of product_clean that are exact repeats of an earlier row.
# Why: the same check as before cleaning, re-run because columns were dropped and added. A repeated row would count the
# same money twice.
# Found: none.
# Decision: nothing to fix.
product_clean.duplicated().sum()

np.int64(0)

In [ ]:
# Counted the rows of country_clean that are exact repeats of an earlier row, as for product_clean.
# Why: the same check as before cleaning, re-run because a row was dropped and columns were added.
# Found: none.
# Decision: nothing to fix.
country_clean.duplicated().sum()

np.int64(0)

In [ ]:
product_clean.head()

,hs4,hs6,cn8,value_gbp,net_mass_kg,flow_type,year,month,date,row_type,product_group
0,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,2,NaN,EU - Exports,2022,January,2022-01-01,estimate,NaN
1,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1350945,NaN,EU - Exports,2022,February,2022-02-01,estimate,NaN
2,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1084760,NaN,EU - Exports,2022,March,2022-03-01,estimate,NaN
3,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1369748,NaN,EU - Exports,2022,April,2022-04-01,estimate,NaN
4,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,30 🔸 Non Response Estimates,1396,NaN,EU - Exports,2022,May,2022-05-01,estimate,NaN


In [ ]:
# Counted the different dates in each flow_type among the below_threshold rows of product_clean.
# Why: check 3 passed for each flow, but it cannot see a gap in one kind of row, because other rows fill every month.
# Found: only the two EU flows have below_threshold rows, and each has one month fewer than the whole table.
# Decision: find out which month is missing in the next cell.
product_clean[product_clean['row_type'] == 'below_threshold'].groupby('flow_type')['date'].nunique()

,date
flow_type,
EU - Exports,54
EU - Imports,54


In [ ]:
# Listed the different dates among the below_threshold rows of product_clean in 2026.
# Why: to find which month is missing from the below_threshold rows.
# Found: the list goes from May straight to July, so June 2026 has no below_threshold rows. It covers both flows together,
# so neither flow has a June line.
# Decision: nothing to fix, and no row is added for June because it is not known what it would hold. No cause is claimed.
# To be noted in the documentation.
product_clean[(product_clean['row_type'] == 'below_threshold') & (product_clean['year'] == 2026)]['date'].unique()

<DatetimeArray>
['2026-01-01 00:00:00', '2026-02-01 00:00:00', '2026-03-01 00:00:00',
 '2026-04-01 00:00:00', '2026-05-01 00:00:00', '2026-07-01 00:00:00']
Length: 6, dtype: datetime64[ns]

In [ ]:
country_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12775 entries, 0 to 12775
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   eu_status    12775 non-null  object        
 1   continent    12775 non-null  object        
 2   country      12775 non-null  object        
 3   value_gbp    12775 non-null  int64         
 4   net_mass_kg  12665 non-null  float64       
 5   flow_type    12775 non-null  object        
 6   year         12775 non-null  int64         
 7   month        12775 non-null  object        
 8   date         12775 non-null  datetime64[ns]
 9   row_type     12775 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(6)
memory usage: 1.1+ MB


### Step 9: find the spikes (check 6)

In [ ]:
# Built summary from country_clean: the median month and the biggest month (max) of value_gbp for each country and flow_type.
# Why: check 6, spikes. A month far above a country's usual month could be a mistake, so each country's biggest month is
# compared with its median month.
summary = country_clean.groupby(['flow_type', 'country'])['value_gbp'].agg(['median', 'max'])

In [ ]:
print(summary)

                                     median        max
flow_type        country                              
EU - Exports     Austria          8525433.0   27435927
                 Belgium        199690333.0  381888271
                 Bulgaria          541835.0    1976985
                 Croatia           118156.0    3042017
                 Cyprus            402219.0     753060
...                                     ...        ...
Non EU - Imports United States  257601802.0  518222147
                 Uruguay             1445.0       1923
                 Uzbekistan          6149.0       9566
                 Venezuela           5154.0       5154
                 Vietnam            10821.0     451057

[362 rows x 2 columns]


In [ ]:
# Added ratio to summary: max divided by median for each row.
# Why: ratio is how many times bigger the biggest month is than the median month, so all rows compare on one scale.
summary['ratio'] = summary['max'] / summary['median']

In [ ]:
# Sorted summary by ratio from biggest to smallest and showed the first 10 rows.
# Why: the biggest ratios are the rows where one month stands out most from the median month.
# Found: Estimates (EU - Imports) and Israel (Non EU - Imports) have the biggest ratios, and most of the other rows in the
# top 10 hold small amounts of money.
# Decision: review only the rows with a large max, in the next cell.
(summary.sort_values('ratio', ascending = False)).head(10)

median        max        ratio
flow_type        country                                           
EU - Imports     Estimates           3229.0   13598229  4211.281821
Non EU - Imports Israel            337835.0  278201892   823.484518
EU - Exports     Estimates          10537.0    5839861   554.224257
Non EU - Imports Surinam             1686.0     528024   313.181495
Non EU - Exports Mauritania          3060.0     893176   291.887582
Non EU - Imports Namibia             5324.0    1510108   283.641623
Non EU - Exports Ivory Coast         8773.0    1319170   150.367035
                 Guinea              7603.0    1116206   146.811259
                 Burkina             3105.0     450000   144.927536
                 Papua New Guinea    3867.0     542887   140.389708

In [ ]:
# Picked the rows of summary where max is above £10m, sorted them by ratio from biggest to smallest, and showed the first 10.
# Why: the unfiltered top 10 above is mostly rows with small amounts, so only rows with a max above £10m are reviewed.
# This is a view only: nothing is deleted from summary or country_clean.
# Found: two rows have a ratio far above all the others: Estimates (EU - Imports) and Israel (Non EU - Imports).
# Decision: open both up in country_clean in the next cells. Nothing is changed here.
summary[summary['max'] > 10000000].sort_values('ratio', ascending=False).head(10)


median        max        ratio
flow_type        country                                        
EU - Imports     Estimates        3229.0   13598229  4211.281821
Non EU - Imports Israel         337835.0  278201892   823.484518
EU - Exports     Greece        1197118.0   24501385    20.466976
Non EU - Exports Ghana         1352304.0   27371680    20.240774
                 South Korea  13828444.0  231915933    16.770935
                 Ukraine        815134.5   12211353    14.980783
Non EU - Imports South Korea   2588678.0   34865014    13.468270
Non EU - Exports Colombia      4661577.0   52169469    11.191378
Non EU - Imports Turkey        1308836.0   11257482     8.601140
EU - Exports     Sweden        2932982.0   18674346     6.367017

In [ ]:
# Picked the rows of country_clean for Israel in Non EU - Imports, sorted them by value_gbp from biggest to smallest, and
# showed the first 10.
# Why: summary showed Israel in Non EU - Imports has a biggest month far above its median month. This shows which months
# those are.
# Found: a group of months in 2022 and one in April 2023 are far above the rest: tens to hundreds of millions of pounds,
# against a median month of hundreds of thousands. The next largest month is only about £2.5m.
# Decision: keep all rows. Nothing here shows whether these months are errors or real, so no cause is claimed.
# Checked in Analyse, Step 5: total imports, the gap, the supplier shares and value per kg were worked out again without the
# eight Israel months. The Non EU line on its own was not given a 12-month average.
country_clean[(country_clean['country'] == 'Israel') & (country_clean['flow_type'] == 'Non EU - Imports')].sort_values('value_gbp', ascending=False).head(10)

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
9175,NON-EU,Middle East and N Africa,Israel,278201892,33131.0,Non EU - Imports,2022,April,2022-04-01,partner
9179,NON-EU,Middle East and N Africa,Israel,219536263,2840.0,Non EU - Imports,2022,August,2022-08-01,partner
9177,NON-EU,Middle East and N Africa,Israel,105149404,6619.0,Non EU - Imports,2022,June,2022-06-01,partner
9182,NON-EU,Middle East and N Africa,Israel,90019127,2578.0,Non EU - Imports,2022,November,2022-11-01,partner
9180,NON-EU,Middle East and N Africa,Israel,88455744,20028.0,Non EU - Imports,2022,September,2022-09-01,partner
9187,NON-EU,Middle East and N Africa,Israel,85719239,1234.0,Non EU - Imports,2023,April,2023-04-01,partner
9176,NON-EU,Middle East and N Africa,Israel,78926845,19202.0,Non EU - Imports,2022,May,2022-05-01,partner
9174,NON-EU,Middle East and N Africa,Israel,77505397,4322.0,Non EU - Imports,2022,March,2022-03-01,partner
9189,NON-EU,Middle East and N Africa,Israel,2514920,7332.0,Non EU - Imports,2023,June,2023-06-01,partner
9194,NON-EU,Middle East and N Africa,Israel,2301795,1278.0,Non EU - Imports,2023,November,2023-11-01,partner


In [ ]:
# Picked the rows of country_clean for Estimates in EU - Imports, sorted them by value_gbp from biggest to smallest, and
# showed the first 10.
# Why: summary showed Estimates in EU - Imports has a biggest month far above its median month. This shows which months
# those are.
# Found: every row shown is an estimate with no weight (net_mass_kg is blank). The largest months are mostly recent (2025
# and 2026).
# Decision: keep all rows. Nothing here suggests they are wrong, and rows that hold money stay in the totals. No cause is
# claimed: this table cannot show why these months are larger.
# Done in Analyse: the country tables use partner rows only (Steps 3 and 4), and value per kg leaves out rows without a
# weight, which includes these (Step 7).
country_clean[(country_clean['country'] == 'Estimates') & (country_clean['flow_type'] == 'EU - Imports')].sort_values('value_gbp', ascending=False).head(10)

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
877,EU,European Union,Estimates,13598229,NaN,EU - Imports,2026,June,2026-06-01,estimate
878,EU,European Union,Estimates,2080500,NaN,EU - Imports,2026,July,2026-07-01,estimate
870,EU,European Union,Estimates,1642126,NaN,EU - Imports,2025,November,2025-11-01,estimate
876,EU,European Union,Estimates,352835,NaN,EU - Imports,2026,May,2026-05-01,estimate
874,EU,European Union,Estimates,294267,NaN,EU - Imports,2026,March,2026-03-01,estimate
867,EU,European Union,Estimates,171177,NaN,EU - Imports,2025,August,2025-08-01,estimate
835,EU,European Union,Estimates,133175,NaN,EU - Imports,2022,December,2022-12-01,estimate
872,EU,European Union,Estimates,101227,NaN,EU - Imports,2026,January,2026-01-01,estimate
864,EU,European Union,Estimates,34944,NaN,EU - Imports,2025,May,2025-05-01,estimate
873,EU,European Union,Estimates,20621,NaN,EU - Imports,2026,February,2026-02-01,estimate


In [ ]:
# Built coverage from country_clean: the biggest month (max) and the total (sum) of value_gbp for each country and flow_type.
# Why: the filtered view of summary only reviewed rows with a max above £10m. coverage adds each row's total, so it is
# possible to say how much of the money sits in the rows that were not reviewed.
coverage = country_clean.groupby(['flow_type', 'country'])['value_gbp'].agg(['max', 'sum'])



In [ ]:
coverage.head(5)

max          sum
flow_type    country                         
EU - Exports Austria    27435927    520528818
             Belgium   381888271  11617446731
             Bulgaria    1976985     32757140
             Croatia     3042017     17944754
             Cyprus       753060     22542677

In [ ]:
summary.head(5)

median        max      ratio
flow_type    country                                    
EU - Exports Austria     8525433.0   27435927   3.218127
             Belgium   199690333.0  381888271   1.912402
             Bulgaria     541835.0    1976985   3.648685
             Croatia      118156.0    3042017  25.745768
             Cyprus       402219.0     753060   1.872264

In [ ]:
# Counted the rows of coverage where max is £10m or under.
# Why: these are the rows that the filtered view of summary did not review. This counts how many there are.
# Found: most of the country and flow combinations are in this group.
(coverage['max'] <= 10000000).sum()

np.int64(294)

In [ ]:
# Added up the sum column for the rows of coverage where max is £10m or under, and called the result unreviewed_value.
# Why: this is how much money sits in the rows that the filtered view of summary did not review. The next steps turn it
# into a share of all the money.
unreviewed_value = (coverage[coverage['max'] <= 10000000])['sum'].sum()

In [ ]:
unreviewed_value

np.int64(5846473926)

In [ ]:
# Added up the sum column of coverage, and called the result all_value.
# Why: this is all the money in country_clean, so unreviewed_value can be shown as a share of it.
all_value = coverage['sum'].sum()

In [ ]:
all_value

np.int64(209718700270)

In [ ]:
# Divided unreviewed_value by all_value, multiplied by 100 so it reads as a percent, rounded to one decimal place, and
# called the result unreviewed_share.
# Why: this is how much of the money sits in the rows that were not reviewed. It goes into the Decision, so a reader can
# see how much the review left out.
unreviewed_share = (unreviewed_value / all_value * 100).round(1)

In [ ]:
# Showed unreviewed_share.
# Why: this is the last step of check 6, and it shows how much of the money the review did not look at.
# Found: the rows that were not reviewed hold a small share of all the money (exports and imports together), and none of
# their months is above £10m.
# Decision: keep all rows. Nothing in check 6 changes country_clean. Rows with a max of £10m or under were not reviewed.
# Done in Analyse: the Israel test is Step 5. Estimates stay out of the country tables and value per kg (Steps 3, 4 and 7).
unreviewed_share

np.float64(2.8)

### Step 10: save the cleaned tables

In [ ]:
# Saved country_clean as a CSV file called country_clean.csv in the Project 3 folder, without the row numbers.
# Why: the Power BI dashboard needs the cleaned table as a file. The original HMRC files stay untouched.
country_clean.to_csv('/content/drive/MyDrive/Project 3/country_clean.csv', index=False)

In [ ]:
# Saved product_clean as a CSV file called product_clean.csv in the Project 3 folder, without the row numbers.
# Why: the Power BI dashboard needs the cleaned table as a file. The original HMRC files stay untouched.
product_clean.to_csv('/content/drive/MyDrive/Project 3/product_clean.csv', index=False)

## Analyse phase

### Step 1: build one row per month

In [ ]:
# Built monthly from country_clean: one row per date, one column per flow type, with value_gbp added up in each cell.
# Why: the questions on exports, imports and the gap all need one value per month for each flow.
monthly = country_clean.pivot_table(values='value_gbp', columns='flow_type', aggfunc='sum', index='date')

In [ ]:
monthly.shape

(55, 4)

In [ ]:
monthly.head(10)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports
date,,,,
2022-01-01,472109643,1154639185,700463239,514876233
2022-02-01,918037338,1301958533,808339491,621125576
2022-03-01,1086990323,1600528807,944113574,805618534
2022-04-01,729387199,1637596367,691283385,795310013
2022-05-01,748633217,1988675442,1139337732,640250255
2022-06-01,922431755,1459687606,1013850556,643888611
2022-07-01,999928067,1406420400,1042371704,639969490
2022-08-01,1140845893,1757648857,985420778,569417189
2022-09-01,1071999622,1584252587,963241780,623025844


In [ ]:
all_value

np.int64(209718700270)

In [ ]:
monthly.sum().sum()

np.int64(209718700270)

In [ ]:
# Added up every cell of monthly and compared the total with all_value, the total of country_clean.
# Why: the trends for questions 1 and 3 are built on monthly, so it has to hold all the money, none lost or counted twice.
# Found: the two totals are equal.
# Decision: keep monthly as built. Nothing to fix.
all_value == monthly.sum().sum()

np.True_

In [ ]:
# Added exports, imports and gap to monthly: exports is EU plus non-EU exports, imports is EU plus non-EU imports, and gap
# is exports minus imports for each month. Below zero, the UK bought more than it sold.
# Why: question 1 is about exports and question 3 is about the gap between exports and imports, so each needs one value
# per month.
monthly['exports'] = monthly['EU - Exports'] + monthly['Non EU - Exports']
monthly['imports'] = monthly['EU - Imports'] + monthly['Non EU - Imports']
monthly['gap'] = monthly['exports'] - monthly['imports']

In [ ]:
monthly.head()

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap
date,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280
2022-03-01,1086990323,1600528807,944113574,805618534,2031103897,2406147341,-375043444
2022-04-01,729387199,1637596367,691283385,795310013,1420670584,2432906380,-1012235796
2022-05-01,748633217,1988675442,1139337732,640250255,1887970949,2628925697,-740954748


In [ ]:
# Added exports and imports together for each month, added up those totals, and compared with all_value.
# Why: .head() only shows a few months. This checks the new columns hold every flow in every month, once.
# Found: the total equals all_value.
# Decision: keep exports, imports and gap as built. Nothing to fix.
(monthly['exports'] + monthly['imports']).sum() == all_value

np.True_

### Step 2: average over 12 months and compare with the year before

In [ ]:
monthly['exports'].rolling(12).mean()

,exports
date,
2022-01-01,NaN
2022-02-01,NaN
2022-03-01,NaN
2022-04-01,NaN
2022-05-01,NaN
2022-06-01,NaN
2022-07-01,NaN
2022-08-01,NaN
2022-09-01,NaN


In [ ]:
# Added exports_12m to monthly: exports averaged over each month and the 11 months before it.
# Why: single months swing, so question 1 is hard to read. 12 months covers every season once.
monthly['exports_12m'] = monthly['exports'].rolling(12).mean()

In [ ]:
monthly.head(14)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,exports_12m
date,,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536,NaN
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280,NaN
2022-03-01,1086990323,1600528807,944113574,805618534,2031103897,2406147341,-375043444,NaN
2022-04-01,729387199,1637596367,691283385,795310013,1420670584,2432906380,-1012235796,NaN
2022-05-01,748633217,1988675442,1139337732,640250255,1887970949,2628925697,-740954748,NaN
2022-06-01,922431755,1459687606,1013850556,643888611,1936282311,2103576217,-167293906,NaN
2022-07-01,999928067,1406420400,1042371704,639969490,2042299771,2046389890,-4090119,NaN
2022-08-01,1140845893,1757648857,985420778,569417189,2126266671,2327066046,-200799375,NaN
2022-09-01,1071999622,1584252587,963241780,623025844,2035241402,2207278431,-172037029,NaN


In [ ]:
# Averaged the first 12 months of exports with a plain mean and compared it with the first average in exports_12m.
# Why: question 1 is read from exports_12m, so its first average has to match a plain average of the same 12 months.
# Found: the two values are equal.
# Decision: keep exports_12m as built. The first 11 months stay NaN because they do not have 12 months to use.
monthly['exports_12m'].loc['2022-12-01'] == monthly['exports'].head(12).mean()

np.True_

In [ ]:
# Added imports_12m to monthly: imports averaged over each month and the 11 months before it.
# Why: question 3 is about the gap between exports and imports, and imports swing from month to month like exports does.
monthly['imports_12m'] = monthly['imports'].rolling(12).mean()

In [ ]:
monthly.head(14)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,exports_12m,imports_12m
date,,,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536,NaN,NaN
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280,NaN,NaN
2022-03-01,1086990323,1600528807,944113574,805618534,2031103897,2406147341,-375043444,NaN,NaN
2022-04-01,729387199,1637596367,691283385,795310013,1420670584,2432906380,-1012235796,NaN,NaN
2022-05-01,748633217,1988675442,1139337732,640250255,1887970949,2628925697,-740954748,NaN,NaN
2022-06-01,922431755,1459687606,1013850556,643888611,1936282311,2103576217,-167293906,NaN,NaN
2022-07-01,999928067,1406420400,1042371704,639969490,2042299771,2046389890,-4090119,NaN,NaN
2022-08-01,1140845893,1757648857,985420778,569417189,2126266671,2327066046,-200799375,NaN,NaN
2022-09-01,1071999622,1584252587,963241780,623025844,2035241402,2207278431,-172037029,NaN,NaN


In [ ]:
# Averaged the first 12 months of imports with a plain mean and compared it with the first average in imports_12m.
# Why: imports_12m feeds question 3, so its first average has to match a plain average of the same 12 months.
# Found: the two values are equal.
# Decision: keep imports_12m as built. The first 11 months stay NaN because they do not have 12 months to use.
monthly['imports_12m'].loc['2022-12-01'] == monthly['imports'].head(12).mean()

np.True_

In [ ]:
# Added gap_12m to monthly: gap averaged over each month and the 11 months before it.
# Why: question 3 is about the gap, and its monthly swings hide whether it is widening or narrowing.
monthly['gap_12m'] = monthly['gap'].rolling(12).mean()

In [ ]:
monthly.head(14)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,exports_12m,imports_12m,gap_12m
date,,,,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536,NaN,NaN,NaN
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280,NaN,NaN,NaN
2022-03-01,1086990323,1600528807,944113574,805618534,2031103897,2406147341,-375043444,NaN,NaN,NaN
2022-04-01,729387199,1637596367,691283385,795310013,1420670584,2432906380,-1012235796,NaN,NaN,NaN
2022-05-01,748633217,1988675442,1139337732,640250255,1887970949,2628925697,-740954748,NaN,NaN,NaN
2022-06-01,922431755,1459687606,1013850556,643888611,1936282311,2103576217,-167293906,NaN,NaN,NaN
2022-07-01,999928067,1406420400,1042371704,639969490,2042299771,2046389890,-4090119,NaN,NaN,NaN
2022-08-01,1140845893,1757648857,985420778,569417189,2126266671,2327066046,-200799375,NaN,NaN,NaN
2022-09-01,1071999622,1584252587,963241780,623025844,2035241402,2207278431,-172037029,NaN,NaN,NaN


In [ ]:
# Averaged the first 12 months of gap with a plain mean and compared it with the first average in gap_12m.
# Why: gap_12m feeds question 3, so its first average has to match a plain average of the same 12 months.
# Found: the two values are equal.
# Decision: keep gap_12m as built. The first 11 months stay NaN because they do not have 12 months to use.
monthly['gap_12m'].loc['2022-12-01'] == monthly['gap'].head(12).mean()

np.True_

In [ ]:
# Added exports_yoy to monthly: the percentage change of exports_12m against the same month a year earlier.
# Why: question 1 asks whether exports are higher or lower than before, and this gives one number to quote for each year.
# Found: the first rows are NaN, because each row needs a 12-month average and the one 12 rows earlier (two years of months).
# Decision: keep exports_yoy as built. Comparing 12-month averages keeps the monthly swings out, and starts a year later.
monthly['exports_yoy'] = monthly['exports_12m'].pct_change(12)

In [ ]:
monthly.head(25)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,exports_12m,imports_12m,gap_12m,exports_yoy
date,,,,,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536,NaN,NaN,NaN,NaN
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280,NaN,NaN,NaN,NaN
2022-03-01,1086990323,1600528807,944113574,805618534,2031103897,2406147341,-375043444,NaN,NaN,NaN,NaN
2022-04-01,729387199,1637596367,691283385,795310013,1420670584,2432906380,-1012235796,NaN,NaN,NaN,NaN
2022-05-01,748633217,1988675442,1139337732,640250255,1887970949,2628925697,-740954748,NaN,NaN,NaN,NaN
2022-06-01,922431755,1459687606,1013850556,643888611,1936282311,2103576217,-167293906,NaN,NaN,NaN,NaN
2022-07-01,999928067,1406420400,1042371704,639969490,2042299771,2046389890,-4090119,NaN,NaN,NaN,NaN
2022-08-01,1140845893,1757648857,985420778,569417189,2126266671,2327066046,-200799375,NaN,NaN,NaN,NaN
2022-09-01,1071999622,1584252587,963241780,623025844,2035241402,2207278431,-172037029,NaN,NaN,NaN,NaN


In [ ]:
# Added imports_yoy to monthly: the percentage change of imports_12m against the same month a year earlier.
# Why: question 3 asks whether the UK leans more on imports, so imports need the same year-on-year view as exports.
# Found: the first rows are NaN for the same reason as exports_yoy, because two years of months are needed.
# Decision: keep imports_yoy as built.
monthly['imports_yoy'] = monthly['imports_12m'].pct_change(12)

In [ ]:
monthly.head(25)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,exports_12m,imports_12m,gap_12m,exports_yoy,imports_yoy
date,,,,,,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536,NaN,NaN,NaN,NaN,NaN
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280,NaN,NaN,NaN,NaN,NaN
2022-03-01,1086990323,1600528807,944113574,805618534,2031103897,2406147341,-375043444,NaN,NaN,NaN,NaN,NaN
2022-04-01,729387199,1637596367,691283385,795310013,1420670584,2432906380,-1012235796,NaN,NaN,NaN,NaN,NaN
2022-05-01,748633217,1988675442,1139337732,640250255,1887970949,2628925697,-740954748,NaN,NaN,NaN,NaN,NaN
2022-06-01,922431755,1459687606,1013850556,643888611,1936282311,2103576217,-167293906,NaN,NaN,NaN,NaN,NaN
2022-07-01,999928067,1406420400,1042371704,639969490,2042299771,2046389890,-4090119,NaN,NaN,NaN,NaN,NaN
2022-08-01,1140845893,1757648857,985420778,569417189,2126266671,2327066046,-200799375,NaN,NaN,NaN,NaN,NaN
2022-09-01,1071999622,1584252587,963241780,623025844,2035241402,2207278431,-172037029,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Added gap_yoy_gbp to monthly: the change in gap_12m, in pounds, against the same month a year earlier.
# Why: the gap crosses zero, so a percentage would mislead. A change in pounds shows whether the gap moved up or down.
# Found: the first rows are NaN, as for exports_yoy. The first number matches the hand calculation.
# Decision: keep gap_yoy_gbp as built. Pounds for the gap, percentages for exports and imports.
monthly['gap_yoy_gbp'] = monthly['gap_12m'].diff(12)

In [ ]:
monthly.head(25)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,exports_12m,imports_12m,gap_12m,exports_yoy,imports_yoy,gap_yoy_gbp
date,,,,,,,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-01,1086990323,1600528807,944113574,805618534,2031103897,2406147341,-375043444,NaN,NaN,NaN,NaN,NaN,NaN
2022-04-01,729387199,1637596367,691283385,795310013,1420670584,2432906380,-1012235796,NaN,NaN,NaN,NaN,NaN,NaN
2022-05-01,748633217,1988675442,1139337732,640250255,1887970949,2628925697,-740954748,NaN,NaN,NaN,NaN,NaN,NaN
2022-06-01,922431755,1459687606,1013850556,643888611,1936282311,2103576217,-167293906,NaN,NaN,NaN,NaN,NaN,NaN
2022-07-01,999928067,1406420400,1042371704,639969490,2042299771,2046389890,-4090119,NaN,NaN,NaN,NaN,NaN,NaN
2022-08-01,1140845893,1757648857,985420778,569417189,2126266671,2327066046,-200799375,NaN,NaN,NaN,NaN,NaN,NaN
2022-09-01,1071999622,1584252587,963241780,623025844,2035241402,2207278431,-172037029,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
monthly.tail(25)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,exports_12m,imports_12m,gap_12m,exports_yoy,imports_yoy,gap_yoy_gbp
date,,,,,,,,,,,,,
2024-07-01,676416136,1489837584,918762956,543050125,1595179092,2032887709,-437708617,1.823337e+09,1.902904e+09,-7.956701e+07,-0.054009,-0.056735,1.035508e+07
2024-08-01,829219209,1274942217,1031265364,402234536,1860484573,1677176753,183307820,1.836727e+09,1.909757e+09,-7.303032e+07,-0.029162,-0.023815,-8.580471e+06
2024-09-01,821974169,1477900338,958721719,560023794,1780695888,2037924132,-257228244,1.856386e+09,1.918618e+09,-6.223190e+07,0.002895,-0.007633,2.011563e+07
2024-10-01,1043269089,1627464790,1075930898,559396830,2119199987,2186861620,-67661633,1.867422e+09,1.929602e+09,-6.217933e+07,0.006174,-0.001260,1.389312e+07
2024-11-01,772028671,1542619978,1006600662,439253325,1778629333,1981873303,-203243970,1.819476e+09,1.943516e+09,-1.240401e+08,-0.023814,0.052204,-1.408116e+08
2024-12-01,661326451,1609213883,1001727535,401696351,1663053986,2010910234,-347856248,1.807423e+09,1.955637e+09,-1.482140e+08,-0.019837,0.066573,-1.586452e+08
2025-01-01,754042962,1188583952,952626815,640516766,1706669777,1829100718,-122430941,1.792581e+09,1.949690e+09,-1.571093e+08,-0.029689,0.060398,-1.658999e+08
2025-02-01,895009778,1259614323,1072377988,453953143,1967387766,1713567466,253820300,1.805333e+09,1.943333e+09,-1.379999e+08,-0.021645,0.061477,-1.524919e+08
2025-03-01,870557176,1499617425,1200358552,457161515,2070915728,1956778940,114136788,1.834117e+09,1.938388e+09,-1.042711e+08,0.003935,0.066655,-1.139405e+08


In [ ]:
# Picked the December rows and the latest month from monthly, with the 12-month averages and the year-on-year changes.
# Why: December rows are calendar-year averages, so they answer questions 1 and 3 side by side without reading every month.
# Found: exports fall from each December to the next, and the latest 12 months fall the most.
# Found: imports dip in 2023, rise in 2024 and 2025, then level off.
# Found: the gap is below zero (the UK buys more than it sells) except in Dec 2023, and it grows each time after that.
# Decision: report calendar years from the December rows, and the latest 12 months from the Jul 2026 row (no December yet).
# Checked in Analyse, Step 5: seven of the eight Israel months fall in the 12 months to Dec 2022. Without them the Dec 2022
# gap is -£261m a month, not -£339m, and its 2023 rise is £278m a month, not £349m.
monthly.loc[['2022-12-01', '2023-12-01', '2024-12-01', '2025-12-01', '2026-07-01'], ['exports_12m', 'exports_yoy', 'imports_12m', 'imports_yoy', 'gap_12m', 'gap_yoy_gbp']]

flow_type,exports_12m,exports_yoy,imports_12m,imports_yoy,gap_12m,gap_yoy_gbp
date,,,,,,
2022-12-01,1.884293e+09,NaN,2.223162e+09,NaN,-3.388688e+08,NaN
2023-12-01,1.844003e+09,-0.021382,1.833572e+09,-0.175242,1.043122e+07,3.493000e+08
2024-12-01,1.807423e+09,-0.019837,1.955637e+09,0.066573,-1.482140e+08,-1.586452e+08
2025-12-01,1.748670e+09,-0.032507,1.994516e+09,0.019881,-2.458468e+08,-9.763274e+07
2026-07-01,1.703242e+09,-0.078767,1.976216e+09,-0.007694,-2.729736e+08,-1.303068e+08


### Step 3: who buys the exports

In [ ]:
# Built export_partners from country_clean: only partner rows, and only the two export flows.
# Why: question 2 asks who buys the exports, so estimate and stores rows (not countries) and import flows stay out.
export_partners = country_clean[(country_clean['row_type'] == 'partner') & (country_clean['flow_type'].isin(['EU - Exports', 'Non EU - Exports']))]

In [ ]:
export_partners.shape

(8741, 10)

In [ ]:
export_partners.head(15)

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
0,EU,European Union,Austria,6814787,48756.0,EU - Exports,2022,January,2022-01-01,partner
1,EU,European Union,Austria,11891585,74395.0,EU - Exports,2022,February,2022-02-01,partner
2,EU,European Union,Austria,7369406,65655.0,EU - Exports,2022,March,2022-03-01,partner
3,EU,European Union,Austria,7723548,64349.0,EU - Exports,2022,April,2022-04-01,partner
4,EU,European Union,Austria,14518099,17780.0,EU - Exports,2022,May,2022-05-01,partner
5,EU,European Union,Austria,8519771,113842.0,EU - Exports,2022,June,2022-06-01,partner
6,EU,European Union,Austria,7156935,49092.0,EU - Exports,2022,July,2022-07-01,partner
7,EU,European Union,Austria,9878791,85432.0,EU - Exports,2022,August,2022-08-01,partner
8,EU,European Union,Austria,7849657,24903.0,EU - Exports,2022,September,2022-09-01,partner
9,EU,European Union,Austria,5018626,49135.0,EU - Exports,2022,October,2022-10-01,partner


In [ ]:
# Divided the money in export_partners by the money in all exports (monthly['exports']).
# Why: the buyer shares are worked out from partner rows only, so those rows have to hold nearly all of the export money.
# Found: the partner rows hold almost all of the export money. What is left out is small.
# Decision: use export_partners as the base for the buyer shares.
export_partners['value_gbp'].sum() / monthly['exports'].sum()

np.float64(0.9998127401713525)

In [ ]:
# Added up value_gbp for each country across all months and sorted from the biggest buyer down.
# Why: question 2 asks who buys the exports, and a ranking of total value shows the main buyers.
export_by_country = export_partners.groupby('country')['value_gbp'].sum().sort_values(ascending=False)

In [ ]:
export_by_country.head(10)

,value_gbp
country,
United States,25114575800
Belgium,11617446731
Germany,10422403373
France,6858466741
Ireland,6608038709
China,4912035371
Netherlands,4357050010
Japan,2764496114
Switzerland,2287439367


In [ ]:
export_by_country.head(15)

,value_gbp
country,
United States,25114575800
Belgium,11617446731
Germany,10422403373
France,6858466741
Ireland,6608038709
China,4912035371
Netherlands,4357050010
Japan,2764496114
Switzerland,2287439367


In [ ]:
export_by_country.shape

(214,)

In [ ]:
# Divided each country's total in export_by_country by the sum of all totals to get each country's share of export value.
# Why: question 2 asks who buys the exports, and a share shows how much of the export money each buyer holds.
# Found: the shares add up to 1, apart from a tiny rounding gap in the last decimal place.
# Found: the United States holds the biggest share, then Belgium and Germany.
# Decision: keep export_share as built. Shares stay as fractions, so 0.25 means 25%.
# Decision: export_share is not saved, so its Series name (value_gbp) stays. The Share phase uses export_share_by_year.
export_share = export_by_country / export_by_country.sum()

In [ ]:
export_share.head(10)

,value_gbp
country,
United States,0.251565
Belgium,0.116369
Germany,0.104398
France,0.068699
Ireland,0.066191
China,0.049202
Netherlands,0.043643
Japan,0.027691
Switzerland,0.022913


In [ ]:
# Added up all the shares in export_share.
# Why: shares of a whole should add up to 1, so this checks that no country is missing or counted twice.
# Found: the sum is 1, apart from a tiny rounding gap in the last decimal place.
export_share.sum()

np.float64(0.9999999999999999)

In [ ]:
# Added up the 5 biggest shares in export_share.
# Why: question 2 asks who buys the exports, and the top 5 show how much of the money a few buyers hold.
# Found: the top 5 buyers hold over half of the export money.
export_share.head(5).sum()

np.float64(0.6072219118756742)

In [ ]:
# Added up the 10 biggest shares in export_share.
# Why: the top 10 show whether the buying is spread wider than the top 5 suggest.
# Found: the top 10 buyers hold about three quarters of the export money.
# Decision: describe the buying as concentrated in a few countries.
# Checked below, by year: the 5 and 10 biggest buyers over the whole period hold 59% to 63% and 75% to 79% of export value.
export_share.head(10).sum()

np.float64(0.7674766604246885)

In [ ]:
export_partners

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
0,EU,European Union,Austria,6814787,48756.0,EU - Exports,2022,January,2022-01-01,partner
1,EU,European Union,Austria,11891585,74395.0,EU - Exports,2022,February,2022-02-01,partner
2,EU,European Union,Austria,7369406,65655.0,EU - Exports,2022,March,2022-03-01,partner
3,EU,European Union,Austria,7723548,64349.0,EU - Exports,2022,April,2022-04-01,partner
4,EU,European Union,Austria,14518099,17780.0,EU - Exports,2022,May,2022-05-01,partner
...,...,...,...,...,...,...,...,...,...,...
12716,NON-EU,Western Europe exc EU,Turkey,13926458,87190.0,Non EU - Exports,2026,March,2026-03-01,partner
12717,NON-EU,Western Europe exc EU,Turkey,25130224,132340.0,Non EU - Exports,2026,April,2026-04-01,partner
12718,NON-EU,Western Europe exc EU,Turkey,35872727,103528.0,Non EU - Exports,2026,May,2026-05-01,partner
12719,NON-EU,Western Europe exc EU,Turkey,57050990,119642.0,Non EU - Exports,2026,June,2026-06-01,partner


In [ ]:
# Built export_by_country_year from export_partners: countries down the side, years across, value_gbp added up in each cell.
# Why: question 2 asks who buys the exports, and one column per year lets us compare the buyers year by year.
# Found: some cells are NaN. Those countries have no export rows in that year, so no exports were recorded to them.
# Decision: keep the NaN cells, in the saved file too. Sums skip them, and NaN means no trade was recorded that year.
export_by_country_year = export_partners.pivot_table(index='country', columns='year', values='value_gbp', aggfunc='sum')

In [ ]:
export_by_country_year

year,2022,2023,2024,2025,2026
country,,,,,
Afghanistan,89927.0,86080.0,518373.0,558524.0,13147.0
Albania,747627.0,1064520.0,1099846.0,1744042.0,639792.0
Algeria,35573342.0,37325579.0,45934535.0,41635110.0,18226916.0
American Samoa,NaN,1696.0,NaN,NaN,NaN
Andorra,163898.0,114016.0,69481.0,38089.0,21512.0
...,...,...,...,...,...
Venezuela,1050267.0,477554.0,664142.0,1189048.0,360402.0
Vietnam,77230646.0,78928794.0,141400370.0,129852076.0,81980872.0
Yemen,469537.0,414220.0,1414673.0,922906.0,183270.0


In [ ]:
# Looked up the United States row in export_by_country_year.
# Why: pivot_table averages unless told to add up, so one known total shows whether the cells are sums.
# Found: the 2022 value matches the total worked out separately, so the cells are sums and not averages.
# Decision: keep export_by_country_year as built.
export_by_country_year.loc['United States']

,United States
year,
2022,5.137054e+09
2023,6.452182e+09
2024,5.042756e+09
2025,5.246201e+09
2026,3.236382e+09


In [ ]:
# Divided export_by_country_year by its column totals to get each country's share of the export value in each year.
# Why: question 2 asks who buys the exports, and a share shows who matters whatever the overall level.
# Found: each year's column adds up to 1, and the United States row matches the figures worked out separately.
# Found: a January to July share can differ from a full-year share by up to 2.1 points for the 5 biggest buyers, so
# 2026 is labelled a part year (see the January to July cell below).
# Decision: keep export_share_by_year as built. Shares stay as fractions.
export_share_by_year = export_by_country_year / export_by_country_year.sum()

In [ ]:
export_share_by_year

year,2022,2023,2024,2025,2026
country,,,,,
Afghanistan,0.000004,3.890095e-06,0.000024,0.000027,0.000001
Albania,0.000033,4.810738e-05,0.000051,0.000083,0.000051
Algeria,0.001574,1.686803e-03,0.002118,0.001984,0.001465
American Samoa,NaN,7.664499e-08,NaN,NaN,NaN
Andorra,0.000007,5.152568e-06,0.000003,0.000002,0.000002
...,...,...,...,...,...
Venezuela,0.000046,2.158144e-05,0.000031,0.000057,0.000029
Vietnam,0.003418,3.566920e-03,0.006520,0.006189,0.006591
Yemen,0.000021,1.871927e-05,0.000065,0.000044,0.000015


In [ ]:
# Added up each year's column of export_share_by_year.
# Why: each year's shares should add up to 1, so this checks that no country is missing or counted twice in any year.
# Found: every year adds up to 1.
export_share_by_year.sum()

,0
year,
2022,1.0
2023,1.0
2024,1.0
2025,1.0
2026,1.0


In [ ]:
# Looked up the United States row in export_share_by_year.
# Why: the United States is the biggest buyer over the whole period, so it is the row to check against the figures worked
# out separately.
# Found: the United States share is lowest in 2022, highest in 2023, and in between in the later years.
export_share_by_year.loc['United States']

,United States
year,
2022,0.227338
2023,0.291585
2024,0.232516
2025,0.250027
2026,0.260193


In [ ]:
# Picked the 10 biggest buyers out of export_share_by_year to read their shares across the years.
# Why: question 2 asks who buys the exports, and the biggest buyers show whether the buyers change from year to year.
# Found: in 2023 the United States share jumped and Germany's fell by a similar amount.
# Found: the United States share went back near its 2022 level in 2024, and Germany's recovered more slowly.
# Decision: report the 2023 swing as it is, and give no cause, because customs data cannot show one.
# Checked month by month at the end of this step: the 2023 swing runs through the year, not one month.
export_share_by_year.loc[['United States', 'Belgium', 'Germany', 'France', 'Ireland', 'China', 'Netherlands', 'Japan', 'Switzerland', 'Australia']]

year,2022,2023,2024,2025,2026
country,,,,,
United States,0.227338,0.291585,0.232516,0.250027,0.260193
Belgium,0.131127,0.108423,0.126498,0.104820,0.105510
Germany,0.130737,0.071671,0.093846,0.107057,0.128684
France,0.051230,0.062433,0.068276,0.083180,0.087894
Ireland,0.071084,0.072048,0.070108,0.063910,0.043898
China,0.053710,0.044015,0.053766,0.049150,0.042373
Netherlands,0.055078,0.045096,0.038784,0.039049,0.036510
Japan,0.026728,0.022620,0.028228,0.032301,0.029750
Switzerland,0.025237,0.033646,0.021895,0.014385,0.015757


In [ ]:
# Added up the shares of the 10 biggest buyers over the whole period, in each year.
# Why: the top 10 show whether the wider group of main buyers holds a steady share too.
# Found: the top 10 hold between three quarters and four fifths of the export money in every year.
# Decision: describe the concentration as steady, even though individual buyers move.
export_share_by_year.loc[['United States',    'Belgium',    'Germany', 'France',    'Ireland',    'China', 'Netherlands',    'Japan', 'Switzerland', 'Australia']].sum()

,0
year,
2022,0.789210
2023,0.768351
2024,0.752644
2025,0.759983
2026,0.764941


In [ ]:
# Added up the shares of the 5 biggest buyers over the whole period, in each year.
# Why: question 2 asks who buys the exports, and the top 5 show whether a few buyers hold the money every year.
# Found: the top 5 hold about three fifths of the export money in every year, with little change.
export_share_by_year.loc[['United States',    'Belgium',    'Germany', 'France',    'Ireland']].sum()

,0
year,
2022,0.611517
2023,0.606160
2024,0.591243
2025,0.608995
2026,0.626178


In [ ]:
# Worked out each month's share of export value for the United States and Germany, and showed 2022 to 2024 as percentages.
# Why: the 2023 swing in export_share_by_year could come from one big month or from most of the year.
# Found: the United States share is above its 2022 share (22.7%) in 11 of the 12 months of 2023, and Germany's is below its
# 2022 share (13.1%) in 11 of 12. The United States share already starts to rise in October 2022.
# Decision: report the 2023 swing as a yearly change, with no cause. It runs through the year, not one month.
export_month_share = export_partners.pivot_table(index='date', columns='country', values='value_gbp', aggfunc='sum')
export_month_share = export_month_share.div(export_month_share.sum(axis=1), axis=0)
export_month_share.loc['2022':'2024', ['United States', 'Germany']].mul(100).round(1)

country,United States,Germany
date,,
2022-01-01,26.2,15.0
2022-02-01,16.1,4.9
2022-03-01,18.5,19.8
2022-04-01,21.4,7.7
2022-05-01,21.5,9.8
2022-06-01,23.3,14.4
2022-07-01,19.4,14.2
2022-08-01,21.7,15.6
2022-09-01,21.6,22.4


### Step 4: who supplies the imports

In [ ]:
# Built import_partners from country_clean: only partner rows, and only the two import flows.
# Why: question 3 asks whether the UK leans more on imports, so estimate rows, stores rows and exports stay out.
import_partners = country_clean[(country_clean['row_type'] == 'partner') & (country_clean['flow_type'].isin(['EU - Imports', 'Non EU - Imports']))]

In [ ]:
import_partners.shape

(3902, 10)

In [ ]:
import_partners.head(15)

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
55,EU,European Union,Austria,13605996,440026.0,EU - Imports,2022,January,2022-01-01,partner
56,EU,European Union,Austria,15042173,524536.0,EU - Imports,2022,February,2022-02-01,partner
57,EU,European Union,Austria,14960585,543850.0,EU - Imports,2022,March,2022-03-01,partner
58,EU,European Union,Austria,20679504,427435.0,EU - Imports,2022,April,2022-04-01,partner
59,EU,European Union,Austria,25073996,649779.0,EU - Imports,2022,May,2022-05-01,partner
60,EU,European Union,Austria,13737932,340007.0,EU - Imports,2022,June,2022-06-01,partner
61,EU,European Union,Austria,14820501,456328.0,EU - Imports,2022,July,2022-07-01,partner
62,EU,European Union,Austria,18938006,508139.0,EU - Imports,2022,August,2022-08-01,partner
63,EU,European Union,Austria,7729400,488248.0,EU - Imports,2022,September,2022-09-01,partner
64,EU,European Union,Austria,19601866,156642.0,EU - Imports,2022,October,2022-10-01,partner


In [ ]:
# Divided the money in import_partners by the money in all imports (monthly['imports']).
# Why: the supplier shares are worked out from partner rows only, so those rows have to hold nearly all of the import money.
# Found: the partner rows hold almost all of the import money. What is left out is small.
# Decision: use import_partners as the base for the supplier shares.
import_partners['value_gbp'].sum() / monthly['imports'].sum()

np.float64(0.9998305157465618)

In [ ]:
# Added up value_gbp for each country across all months and sorted from the biggest supplier down.
# Why: question 3 asks whether the UK leans more on imports, and a ranking of total value shows who the main suppliers are.
# Found: HMRC records each import under its country of dispatch (where it was sent from), which may not be where it was made.
# Decision: label these countries as countries of dispatch, not makers.
import_by_country = import_partners.groupby('country')['value_gbp'].sum().sort_values(ascending=False)

In [ ]:
import_by_country.head(10)

,value_gbp
country,
Belgium,15014921423
United States,14185629499
Netherlands,13841851134
Germany,13127353871
Ireland,8606218629
Switzerland,8406382534
France,6480697505
Italy,6458177141
Denmark,3947894794


In [ ]:
import_by_country.head(15)

,value_gbp
country,
Belgium,15014921423
United States,14185629499
Netherlands,13841851134
Germany,13127353871
Ireland,8606218629
Switzerland,8406382534
France,6480697505
Italy,6458177141
Denmark,3947894794


In [ ]:
import_by_country.shape

(143,)

In [ ]:
# Divided each country's total in import_by_country by the sum of all totals to get each country's share of import value.
# Why: question 3 asks whether the UK leans more on imports, and a share shows how much each supplier holds.
# Found: the shares add up to 1.
# Found: Belgium, the United States, the Netherlands and Germany hold similar shares, each about an eighth.
# Decision: keep import_share as built. Shares stay as fractions, so 0.25 means 25%.
# Decision: import_share is not saved, so its Series name (value_gbp) stays. The Share phase uses import_share_by_year.
import_share = import_by_country / import_by_country.sum()

In [ ]:
import_share.head(10)

,value_gbp
country,
Belgium,0.136688
United States,0.129139
Netherlands,0.126009
Germany,0.119505
Ireland,0.078347
Switzerland,0.076527
France,0.058997
Italy,0.058792
Denmark,0.035940


In [ ]:
# Added up all the shares in import_share.
# Why: shares of a whole should add up to 1, so this checks that no country is missing or counted twice.
# Found: the sum is 1.
import_share.sum()

np.float64(1.0)

In [ ]:
# Added up the 5 biggest shares in import_share.
# Why: question 3 asks whether the UK leans more on imports, and the top 5 show how much a few suppliers hold.
# Found: the top 5 suppliers hold about three fifths of the import money.
import_share.head(5).sum()

np.float64(0.5896865823283101)

In [ ]:
# Added up the 10 biggest shares in import_share.
# Why: the top 10 show whether the supply is spread wider than the top 5 suggest.
# Found: the top 10 suppliers hold over four fifths of the import money.
# Found: that is a bigger share than the top 10 buyers hold of the export money.
# Decision: describe the supply as concentrated in a few countries.
# Checked below, by year: the 5 and 10 biggest suppliers over the whole period hold 50% to 64% and 84% to 86% of import value.
import_share.head(10).sum()

np.float64(0.8526158978117444)

In [ ]:
import_partners

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
55,EU,European Union,Austria,13605996,440026.0,EU - Imports,2022,January,2022-01-01,partner
56,EU,European Union,Austria,15042173,524536.0,EU - Imports,2022,February,2022-02-01,partner
57,EU,European Union,Austria,14960585,543850.0,EU - Imports,2022,March,2022-03-01,partner
58,EU,European Union,Austria,20679504,427435.0,EU - Imports,2022,April,2022-04-01,partner
59,EU,European Union,Austria,25073996,649779.0,EU - Imports,2022,May,2022-05-01,partner
...,...,...,...,...,...,...,...,...,...,...
12771,NON-EU,Western Europe exc EU,Turkey,1490026,129853.0,Non EU - Imports,2026,March,2026-03-01,partner
12772,NON-EU,Western Europe exc EU,Turkey,1840808,129637.0,Non EU - Imports,2026,April,2026-04-01,partner
12773,NON-EU,Western Europe exc EU,Turkey,1167245,61088.0,Non EU - Imports,2026,May,2026-05-01,partner
12774,NON-EU,Western Europe exc EU,Turkey,1371726,111984.0,Non EU - Imports,2026,June,2026-06-01,partner


In [ ]:
# Built import_by_country_year from import_partners: countries down the side, years across, value_gbp added up in each cell.
# Why: question 3 asks whether the UK leans more on imports, and one column per year lets us compare suppliers by year.
# Found: some cells are NaN. Those countries have no import rows in that year, so no imports were recorded from them.
# Decision: keep the NaN cells, in the saved file too. Sums skip them, and NaN means no trade was recorded that year.
import_by_country_year = import_partners.pivot_table(index='country', columns='year', values='value_gbp', aggfunc='sum')

In [ ]:
import_by_country_year

year,2022,2023,2024,2025,2026
country,,,,,
Afghanistan,NaN,NaN,NaN,3.875000e+03,NaN
Albania,1.937000e+03,NaN,NaN,1.512900e+04,NaN
Algeria,9.798200e+04,1.066000e+03,NaN,NaN,NaN
Angola,NaN,NaN,NaN,2.393100e+05,NaN
Antarctica,NaN,NaN,NaN,NaN,1.050000e+03
...,...,...,...,...,...
United States,3.845785e+09,3.425873e+09,3.626320e+09,2.237256e+09,1.050396e+09
Uruguay,NaN,NaN,9.670000e+02,NaN,1.923000e+03
Uzbekistan,NaN,NaN,NaN,9.566000e+03,2.732000e+03


In [ ]:
# Looked up the Belgium row in import_by_country_year.
# Why: pivot_table averages unless told to add up, so one known total shows whether the cells are sums.
# Found: the 2022 value matches the total worked out separately, so the cells are sums and not averages.
# Decision: keep import_by_country_year as built.
import_by_country_year.loc['Belgium']

,Belgium
year,
2022,4.105029e+09
2023,3.373142e+09
2024,3.845155e+09
2025,2.315286e+09
2026,1.376308e+09


In [ ]:
# Divided import_by_country_year by its column totals to get each country's share of the import value in each year.
# Why: question 3 asks whether the UK leans more on imports, and a share shows who matters whatever the overall level.
# Found: each year's column adds up to 1, and the Belgium row matches the figures worked out separately.
# Found: a January to July share can differ from a full-year share by up to 4.5 points for the 5 biggest suppliers, so
# 2026 is labelled a part year (see the January to July cell below).
# Decision: keep import_share_by_year as built. Shares stay as fractions.
# Checked in Analyse, Step 5: without the eight Israel months the supplier shares move only in 2022 and 2023. Israel's share
# of import value in 2022 falls from 3.5% to 0.02%.
import_share_by_year = import_by_country_year / import_by_country_year.sum()

In [ ]:
import_share_by_year

year,2022,2023,2024,2025,2026
country,,,,,
Afghanistan,NaN,NaN,NaN,1.619153e-07,NaN
Albania,7.260730e-08,NaN,NaN,6.321590e-07,NaN
Algeria,3.672797e-06,4.844826e-08,NaN,NaN,NaN
Angola,NaN,NaN,NaN,9.999469e-06,NaN
Antarctica,NaN,NaN,NaN,NaN,7.626575e-08
...,...,...,...,...,...
United States,1.441570e-01,1.557013e-01,1.545245e-01,9.348283e-02,7.629450e-02
Uruguay,NaN,NaN,4.120573e-08,NaN,1.396753e-07
Uzbekistan,NaN,NaN,NaN,3.997113e-07,1.984362e-07


In [ ]:
# Added up each year's column of import_share_by_year.
# Why: each year's shares should add up to 1, so this checks that no country is missing or counted twice in any year.
# Found: every year adds up to 1.
import_share_by_year.sum()

,0
year,
2022,1.0
2023,1.0
2024,1.0
2025,1.0
2026,1.0


In [ ]:
# Looked up the Belgium row in import_share_by_year.
# Why: Belgium is the biggest supplier over the whole period, so it is the row to check against the figures worked out
# separately.
# Found: the Belgium share is steady from 2022 to 2024, then drops sharply in 2025 and stays low in 2026.
import_share_by_year.loc['Belgium']

,Belgium
year,
2022,0.153875
2023,0.153305
2024,0.163849
2025,0.096743
2026,0.099967


In [ ]:
# Picked the 10 biggest suppliers out of import_share_by_year to read their shares across the years.
# Why: question 3 asks whether the UK leans more on imports, and the biggest suppliers show whether they change by year.
# Found: in 2025 the Belgium and United States shares dropped by a similar amount, while France and Spain jumped.
# Found: in 2026 France and Spain fell back, Belgium and the United States stayed low, and Ireland's share kept rising.
# Decision: report the 2025 swing as it is, and give no cause, because customs data cannot show one.
# Checked month by month at the end of this step: the 2025 swing runs through the year, not one month.
import_share_by_year.loc[['Belgium', 'United States', 'Netherlands', 'Germany', 'Ireland', 'Switzerland', 'France', 'Italy', 'Denmark', 'Spain']]

year,2022,2023,2024,2025,2026
country,,,,,
Belgium,0.153875,0.153305,0.163849,0.096743,0.099967
United States,0.144157,0.155701,0.154524,0.093483,0.076294
Netherlands,0.158266,0.119756,0.112812,0.111452,0.121298
Germany,0.127795,0.121486,0.108850,0.112147,0.131225
Ireland,0.053879,0.063339,0.069225,0.087397,0.149558
Switzerland,0.061251,0.076022,0.072509,0.085904,0.097486
France,0.042701,0.055273,0.052087,0.091068,0.052556
Italy,0.076414,0.048131,0.065380,0.057785,0.032204
Denmark,0.026765,0.033570,0.036543,0.034836,0.058396


In [ ]:
# Added up the shares of the 10 biggest suppliers over the whole period, in each year.
# Why: the top 10 show whether the wider group of main suppliers holds a steady share too.
# Found: the top 10 hold a little over four fifths of the import money in every year, easing slightly in 2025 and 2026.
# Decision: describe the concentration as high in every year. Whether it holds steady depends on the spike rows (see below).
import_share_by_year.loc[['Belgium',	'United States',	'Netherlands', 'Germany',	'Ireland',	'Switzerland', 'France',	'Italy', 'Denmark', 'Spain']].sum()

,0
year,
2022,0.862353
2023,0.851459
2024,0.862484
2025,0.840682
2026,0.839521


In [ ]:
# Added up the shares of the 5 biggest suppliers over the whole period, in each year.
# Why: question 3 asks whether the UK leans more on imports, and the top 5 show if a few suppliers hold the money each year.
# Found: the top 5 hold between three fifths and two thirds of the import money in 2022 to 2024, about half in 2025,
# and more again in 2026.
import_share_by_year.loc[['Belgium',	'United States',	'Netherlands', 'Germany',	'Ireland']].sum()

,0
year,
2022,0.637971
2023,0.613587
2024,0.609261
2025,0.501222
2026,0.578343


In [ ]:
# Worked out each month's share of import value for Belgium, the United States, France and Spain, and showed 2024 to 2026
# as percentages.
# Why: the 2025 swing in import_share_by_year could come from one big month or from most of the year.
# Found: Belgium's share is between 7.3% and 12.3% in every month of 2025, against 16.4% for 2024 as a whole.
# Found: France is above its 2024 share (5.2%) in all 12 months. Spain is above its 2024 share (2.7%) in 11 of 12: it rises
# from March, peaks in May (12.7%) and is back at 2.4% in December.
# Found: the United States is below its 2024 share (15.5%) in 11 of 12 months. It is mostly low from August 2024 on.
# Decision: report the 2025 swing as a yearly change, with no cause. It runs through the year, not one month.
import_month_share = import_partners.pivot_table(index='date', columns='country', values='value_gbp', aggfunc='sum')
import_month_share = import_month_share.div(import_month_share.sum(axis=1), axis=0)
import_month_share.loc['2024':'2026', ['Belgium', 'United States', 'France', 'Spain']].mul(100).round(1)

country,Belgium,United States,France,Spain
date,,,,
2024-01-01,12.9,25.1,4.4,2.1
2024-02-01,11.6,17.3,5.0,2.5
2024-03-01,13.9,21.8,4.1,2.4
2024-04-01,11.6,25.9,4.9,2.0
2024-05-01,10.2,16.9,3.8,2.0
2024-06-01,17.6,19.3,4.8,2.6
2024-07-01,23.7,13.5,5.8,3.1
2024-08-01,19.6,7.2,6.7,3.0
2024-09-01,26.8,15.1,6.0,2.9


#### January to July check on the shares

In [ ]:
# Worked out the export and import shares again from January to July of each year only, took the full-year shares away from
# them, and kept the 5 biggest buyers and the 5 biggest suppliers. The table is in percentage points.
# Why: 2026 holds January to July only. If trade is seasonal, a January to July share is not like for like with a
# full-year share.
# Found: the 2026 column is 0.0, because 2026 already is January to July.
# Found: in the other years a January to July share differs from the full-year share by up to 2.1 points for the biggest
# buyers (United States, 2022) and up to 4.5 points for the biggest suppliers (United States, 2024).
# Decision: label 2026 as a part year in the Share phase, shares included.
export_jan_jul = export_partners[export_partners['date'].dt.month <= 7].pivot_table(index='country', columns='year', values='value_gbp', aggfunc='sum')
import_jan_jul = import_partners[import_partners['date'].dt.month <= 7].pivot_table(index='country', columns='year', values='value_gbp', aggfunc='sum')
export_share_jan_jul = export_jan_jul / export_jan_jul.sum()
import_share_jan_jul = import_jan_jul / import_jan_jul.sum()
pd.concat({'exports': (export_share_jan_jul - export_share_by_year).loc[export_share.head(5).index], 'imports': (import_share_jan_jul - import_share_by_year).loc[import_share.head(5).index]}).mul(100).round(1)

year                   2022  2023  2024  2025  2026
        country                                    
exports United States  -2.1  -1.3   0.4   1.0   0.0
        Belgium         0.4   0.2   0.1  -0.4   0.0
        Germany        -0.6  -0.2  -0.3   0.2   0.0
        France          0.5  -0.1   0.2  -0.7   0.0
        Ireland        -1.2  -0.8   0.2   1.0   0.0
imports Belgium        -2.5  -0.2  -1.7  -0.4   0.0
        United States   1.2  -0.1   4.5  -0.1   0.0
        Netherlands    -0.5   0.6   0.2  -0.5   0.0
        Germany         1.2   0.4   0.6  -0.2   0.0
        Ireland         0.4  -0.4  -0.2   0.8   0.0

### Step 5: test whether eight unusual months drive the result

In [ ]:
# Picked out of country_clean the Israel rows in Non EU - Imports with a value_gbp of 10000000 (£10m) or more.
# Why: check 6 found these months far above Israel's other months and flagged them for Analyse to test.
# Found: the £10m cut-off picks out the spike months from check 6, in 2022 and April 2023.
# Decision: keep country_clean as it is. The spike rows are only left out of a copy, in the next steps.
israel_spikes = country_clean[(country_clean['country'] == 'Israel') & (country_clean['value_gbp'] >= 10000000) & (country_clean['flow_type'] == 'Non EU - Imports')]

In [ ]:
israel_spikes.shape

(8, 10)

In [ ]:
israel_spikes.sort_values('value_gbp', ascending=False)

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
9175,NON-EU,Middle East and N Africa,Israel,278201892,33131.0,Non EU - Imports,2022,April,2022-04-01,partner
9179,NON-EU,Middle East and N Africa,Israel,219536263,2840.0,Non EU - Imports,2022,August,2022-08-01,partner
9177,NON-EU,Middle East and N Africa,Israel,105149404,6619.0,Non EU - Imports,2022,June,2022-06-01,partner
9182,NON-EU,Middle East and N Africa,Israel,90019127,2578.0,Non EU - Imports,2022,November,2022-11-01,partner
9180,NON-EU,Middle East and N Africa,Israel,88455744,20028.0,Non EU - Imports,2022,September,2022-09-01,partner
9187,NON-EU,Middle East and N Africa,Israel,85719239,1234.0,Non EU - Imports,2023,April,2023-04-01,partner
9176,NON-EU,Middle East and N Africa,Israel,78926845,19202.0,Non EU - Imports,2022,May,2022-05-01,partner
9174,NON-EU,Middle East and N Africa,Israel,77505397,4322.0,Non EU - Imports,2022,March,2022-03-01,partner


In [ ]:
israel_spikes.index

Index([9174, 9175, 9176, 9177, 9179, 9180, 9182, 9187], dtype='int64')

In [ ]:
# Built country_no_israel_spikes: country_clean with the israel_spikes rows dropped, by their index labels.
# Why: check 6 flagged the spike months for Analyse, so this copy shows the results without them.
# Decision: keep country_clean whole. The name means without the spike rows, not without Israel.
country_no_israel_spikes = country_clean.drop(israel_spikes.index)

In [ ]:
country_no_israel_spikes.shape

(12767, 10)

In [ ]:
country_no_israel_spikes.head(5)

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
0,EU,European Union,Austria,6814787,48756.0,EU - Exports,2022,January,2022-01-01,partner
1,EU,European Union,Austria,11891585,74395.0,EU - Exports,2022,February,2022-02-01,partner
2,EU,European Union,Austria,7369406,65655.0,EU - Exports,2022,March,2022-03-01,partner
3,EU,European Union,Austria,7723548,64349.0,EU - Exports,2022,April,2022-04-01,partner
4,EU,European Union,Austria,14518099,17780.0,EU - Exports,2022,May,2022-05-01,partner


In [ ]:
country_clean.shape


(12775, 10)

In [ ]:
# Added up value_gbp in israel_spikes.
# Why: to know how much money the spike rows hold, so the check in the next cell can compare it with what the drop removes.
# Found: just over £1bn in total.
israel_spikes['value_gbp'].sum()

np.int64(1023513911)

In [ ]:
# Checked that the money in country_clean minus the money in country_no_israel_spikes equals the money in israel_spikes.
# Why: the drop should take out the spike rows and nothing else.
# Found: the check returned True, so the money dropped is exactly the money in the spike rows.
# Decision: use country_no_israel_spikes for the test.
(country_clean['value_gbp'].sum()) - (country_no_israel_spikes['value_gbp'].sum()) == (israel_spikes['value_gbp'].sum())

np.True_

In [ ]:
monthly.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,exports_12m,imports_12m,gap_12m,exports_yoy,imports_yoy,gap_yoy_gbp
date,,,,,,,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-01,1086990323,1600528807,944113574,805618534,2031103897,2406147341,-375043444,NaN,NaN,NaN,NaN,NaN,NaN
2022-04-01,729387199,1637596367,691283385,795310013,1420670584,2432906380,-1012235796,NaN,NaN,NaN,NaN,NaN,NaN
2022-05-01,748633217,1988675442,1139337732,640250255,1887970949,2628925697,-740954748,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Built monthly_no_israel_spikes from country_no_israel_spikes, the same way monthly was built from country_clean.
# Why: question 3 uses imports and the gap, so the same monthly table without the spike rows can be set beside monthly.
monthly_no_israel_spikes = country_no_israel_spikes.pivot_table(values='value_gbp', columns='flow_type', aggfunc='sum', index='date')

In [ ]:
monthly_no_israel_spikes.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports
date,,,,
2022-01-01,472109643,1154639185,700463239,514876233
2022-02-01,918037338,1301958533,808339491,621125576
2022-03-01,1086990323,1600528807,944113574,728113137
2022-04-01,729387199,1637596367,691283385,517108121
2022-05-01,748633217,1988675442,1139337732,561323410


In [ ]:
# Checked that the Non EU - Imports total in monthly minus the same total in monthly_no_israel_spikes
# equals the money in israel_spikes.
# Why: the first check covered every flow, so this one shows the whole drop landed in the Non EU - Imports column.
# Found: the check returned True.
(monthly['Non EU - Imports'].sum()) - (monthly_no_israel_spikes['Non EU - Imports'].sum()) == (israel_spikes['value_gbp'].sum())

np.True_

In [ ]:
# Worked out monthly_no_israel_spikes minus monthly for Non EU - Imports, as spike_diff.
# Why: the totals match, and this shows which months lost money and how much.
spike_diff = (monthly_no_israel_spikes['Non EU - Imports']) - (monthly['Non EU - Imports'])

In [ ]:
spike_diff

,Non EU - Imports
date,
2022-01-01,0
2022-02-01,0
2022-03-01,-77505397
2022-04-01,-278201892
2022-05-01,-78926845
2022-06-01,-105149404
2022-07-01,0
2022-08-01,-219536263
2022-09-01,-88455744


In [ ]:
# Showed only the months in spike_diff where the change is not zero.
# Why: the months that changed should be exactly the spike months, each by the amount in its israel_spikes row.
# Found: only the spike months changed, each by the amount in its israel_spikes row. Every other month is unchanged.
# Decision: continue with monthly_no_israel_spikes.
spike_diff[spike_diff != 0]

,Non EU - Imports
date,
2022-03-01,-77505397
2022-04-01,-278201892
2022-05-01,-78926845
2022-06-01,-105149404
2022-08-01,-219536263
2022-09-01,-88455744
2022-11-01,-90019127
2023-04-01,-85719239


In [ ]:
# Added exports, imports and gap to monthly_no_israel_spikes, built the same way as in monthly.
# Why: question 3 is about imports and the gap, so the test needs the same columns as monthly.
monthly_no_israel_spikes['exports'] = monthly_no_israel_spikes['EU - Exports'] + monthly_no_israel_spikes['Non EU - Exports']
monthly_no_israel_spikes['imports'] = monthly_no_israel_spikes['EU - Imports'] + monthly_no_israel_spikes['Non EU - Imports']
monthly_no_israel_spikes['gap'] = monthly_no_israel_spikes['exports'] - monthly_no_israel_spikes['imports']

In [ ]:
# Added imports_12m, gap_12m, imports_yoy and gap_yoy_gbp to monthly_no_israel_spikes, built the same way as in monthly.
# Why: these are the figures behind question 3. Exports are left out because the spike rows are imports, so exports
# cannot change.
monthly_no_israel_spikes['imports_12m'] = monthly_no_israel_spikes['imports'].rolling(12).mean()
monthly_no_israel_spikes['gap_12m'] = monthly_no_israel_spikes['gap'].rolling(12).mean()
monthly_no_israel_spikes['imports_yoy'] = monthly_no_israel_spikes['imports_12m'].pct_change(12)
monthly_no_israel_spikes['gap_yoy_gbp'] = monthly_no_israel_spikes['gap_12m'].diff(12)

In [ ]:
# Picked the December rows and the latest month from monthly_no_israel_spikes, with the 12-month averages
# and the year-on-year changes.
# Why: these are the same rows picked from monthly earlier, so the two tables can be compared line by line.
monthly_no_israel_spikes.loc[['2022-12-01', '2023-12-01', '2024-12-01', '2025-12-01', '2026-07-01'], ['imports_12m', 'imports_yoy', 'gap_12m', 'gap_yoy_gbp']]

flow_type,imports_12m,imports_yoy,gap_12m,gap_yoy_gbp
date,,,,
2022-12-01,2.145012e+09,NaN,-2.607193e+08,NaN
2023-12-01,1.826428e+09,-0.148523,1.757449e+07,2.782937e+08
2024-12-01,1.955637e+09,0.070744,-1.482140e+08,-1.657885e+08
2025-12-01,1.994516e+09,0.019881,-2.458468e+08,-9.763274e+07
2026-07-01,1.976216e+09,-0.007694,-2.729736e+08,-1.303068e+08


In [ ]:
# Picked the same rows and columns from monthly, which still has the spike rows in, to set beside the table above.
# Why: the difference between the two tables is the effect of the spike rows.
# Found: the 2023 fall in imports is about a seventh without the spike rows and about a sixth with them.
# Found: the 2023 rise in the gap is about a fifth smaller without them, but it is still a rise.
# Found: every change goes the same way in both tables. 2024 differs only a little and 2025 onwards is identical.
# Decision: keep the spike months in the main figures, because they are in the customs data and no cause can be shown.
# Decision: quote the figure without them beside the 2023 fall in imports and the 2023 rise in the gap.
monthly.loc[['2022-12-01', '2023-12-01', '2024-12-01', '2025-12-01', '2026-07-01'], ['imports_12m', 'imports_yoy', 'gap_12m', 'gap_yoy_gbp']]

flow_type,imports_12m,imports_yoy,gap_12m,gap_yoy_gbp
date,,,,
2022-12-01,2.223162e+09,NaN,-3.388688e+08,NaN
2023-12-01,1.833572e+09,-0.175242,1.043122e+07,3.493000e+08
2024-12-01,1.955637e+09,0.066573,-1.482140e+08,-1.586452e+08
2025-12-01,1.994516e+09,0.019881,-2.458468e+08,-9.763274e+07
2026-07-01,1.976216e+09,-0.007694,-2.729736e+08,-1.303068e+08


In [ ]:
monthly_no_israel_spikes

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,imports_12m,gap_12m,imports_yoy,gap_yoy_gbp
date,,,,,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536,NaN,NaN,NaN,NaN
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280,NaN,NaN,NaN,NaN
2022-03-01,1086990323,1600528807,944113574,728113137,2031103897,2328641944,-297538047,NaN,NaN,NaN,NaN
2022-04-01,729387199,1637596367,691283385,517108121,1420670584,2154704488,-734033904,NaN,NaN,NaN,NaN
2022-05-01,748633217,1988675442,1139337732,561323410,1887970949,2549998852,-662027903,NaN,NaN,NaN,NaN
2022-06-01,922431755,1459687606,1013850556,538739207,1936282311,1998426813,-62144502,NaN,NaN,NaN,NaN
2022-07-01,999928067,1406420400,1042371704,639969490,2042299771,2046389890,-4090119,NaN,NaN,NaN,NaN
2022-08-01,1140845893,1757648857,985420778,349880926,2126266671,2107529783,18736888,NaN,NaN,NaN,NaN
2022-09-01,1071999622,1584252587,963241780,534570100,2035241402,2118822687,-83581285,NaN,NaN,NaN,NaN


In [ ]:
# Built import_partners_no_israel_spikes from country_no_israel_spikes: only partner rows, and only the two import flows.
# Why: same filter as import_partners, so the supplier shares can be compared with and without the spike rows.
import_partners_no_israel_spikes = country_no_israel_spikes[(country_no_israel_spikes['row_type'] == 'partner') & (country_no_israel_spikes['flow_type'].isin(['EU - Imports', 'Non EU - Imports']))]

In [ ]:
import_partners_no_israel_spikes

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
55,EU,European Union,Austria,13605996,440026.0,EU - Imports,2022,January,2022-01-01,partner
56,EU,European Union,Austria,15042173,524536.0,EU - Imports,2022,February,2022-02-01,partner
57,EU,European Union,Austria,14960585,543850.0,EU - Imports,2022,March,2022-03-01,partner
58,EU,European Union,Austria,20679504,427435.0,EU - Imports,2022,April,2022-04-01,partner
59,EU,European Union,Austria,25073996,649779.0,EU - Imports,2022,May,2022-05-01,partner
...,...,...,...,...,...,...,...,...,...,...
12771,NON-EU,Western Europe exc EU,Turkey,1490026,129853.0,Non EU - Imports,2026,March,2026-03-01,partner
12772,NON-EU,Western Europe exc EU,Turkey,1840808,129637.0,Non EU - Imports,2026,April,2026-04-01,partner
12773,NON-EU,Western Europe exc EU,Turkey,1167245,61088.0,Non EU - Imports,2026,May,2026-05-01,partner
12774,NON-EU,Western Europe exc EU,Turkey,1371726,111984.0,Non EU - Imports,2026,June,2026-06-01,partner


In [ ]:
# Built import_by_country_year_no_israel_spikes from import_partners_no_israel_spikes, as import_by_country_year was.
# Why: the supplier shares by year come from this table, so they can be compared with and without the spike rows.
import_by_country_year_no_israel_spikes = import_partners_no_israel_spikes.pivot_table(index='country', columns='year', values='value_gbp', aggfunc='sum')

In [ ]:
import_by_country_year_no_israel_spikes

year,2022,2023,2024,2025,2026
country,,,,,
Afghanistan,NaN,NaN,NaN,3.875000e+03,NaN
Albania,1.937000e+03,NaN,NaN,1.512900e+04,NaN
Algeria,9.798200e+04,1.066000e+03,NaN,NaN,NaN
Angola,NaN,NaN,NaN,2.393100e+05,NaN
Antarctica,NaN,NaN,NaN,NaN,1.050000e+03
...,...,...,...,...,...
United States,3.845785e+09,3.425873e+09,3.626320e+09,2.237256e+09,1.050396e+09
Uruguay,NaN,NaN,9.670000e+02,NaN,1.923000e+03
Uzbekistan,NaN,NaN,NaN,9.566000e+03,2.732000e+03


In [ ]:
# Worked out the year totals of import_by_country_year minus those of import_by_country_year_no_israel_spikes.
# Why: each year should lose the money in its spike rows, and a year with no spike rows should lose nothing.
# Found: only 2022 and 2023 lost money, and the amounts match the spike rows of those years. 2024 onwards lost nothing.
# Decision: continue with import_by_country_year_no_israel_spikes.
(import_by_country_year.sum()) - (import_by_country_year_no_israel_spikes.sum())

,0
year,
2022,937794672.0
2023,85719239.0
2024,0.0
2025,0.0
2026,0.0


In [ ]:
# Divided import_by_country_year_no_israel_spikes by its column totals to get each country's share of the import value
# in each year.
# Why: the shares can then be set beside import_share_by_year. A share is a part divided by the yearly total, so dropping
# the spike rows changes the total, and with it every country's share, not only Israel's.
import_share_by_year_no_israel_spikes = import_by_country_year_no_israel_spikes / import_by_country_year_no_israel_spikes.sum()

In [ ]:
# Looked up the Israel row in import_share_by_year.
# Why: the spike rows sit in Israel's imports, so its row shows how much they move its share. The next cell shows the same
# row without them.
# Found: Israel's share is far larger in 2022 than in any later year, and 2023 is the next largest.
import_share_by_year.loc['Israel']

,Israel
year,
2022,0.035362
2023,0.004226
2024,0.000132
2025,0.000300
2026,0.000201


In [ ]:
# Looked up the Israel row in import_share_by_year_no_israel_spikes, to set beside the Israel row in import_share_by_year.
# Why: Israel is where the spike rows sit, so its row shows how much they moved its share.
# Found: Israel's 2022 share is about 0.035 with the spike rows and about 0.0002 without them. 2023 shows the same, smaller.
# Found: from 2024 on the two rows are identical, because those years have no spike rows.
# Decision: keep the spike rows in the supplier shares, and give both figures for 2022,
# because the spike months change the yearly total.
import_share_by_year_no_israel_spikes.loc['Israel']

,Israel
year,
2022,0.000217
2023,0.000332
2024,0.000132
2025,0.000300
2026,0.000201


In [ ]:
# Added up value_gbp for each country in import_partners_no_israel_spikes and sorted from the biggest supplier down.
# Why: the top 10 and top 5 names come from this table, instead of being typed in from the table with the spike rows.
import_by_country_no_israel_spikes = import_partners_no_israel_spikes.groupby('country')['value_gbp'].sum().sort_values(ascending=False)

In [ ]:
import_by_country_no_israel_spikes.head(10)

,value_gbp
country,
Belgium,15014921423
United States,14185629499
Netherlands,13841851134
Germany,13127353871
Ireland,8606218629
Switzerland,8406382534
France,6480697505
Italy,6458177141
Denmark,3947894794


In [ ]:
# Picked the 10 biggest suppliers from import_by_country_no_israel_spikes and added up their shares in each year.
# Why: this sets the top 10 beside the top 10 from import_share_by_year, to see if the spike rows change the picture.
# Found: the 10 biggest suppliers are the same countries, in the same order, with the same money as before.
# Found: only their shares move, because the yearly total is smaller without the spike rows. 2022 rises from a little over
# four fifths to about nine tenths, 2023 barely moves, and 2024 onwards is identical.
# Decision: describe the concentration as high in every year. It eases a little over the period,
# and more so without the spike rows.
import_share_by_year_no_israel_spikes.loc[import_by_country_no_israel_spikes.head(10).index].sum()

,0
year,
2022,0.893771
2023,0.854789
2024,0.862484
2025,0.840682
2026,0.839521


In [ ]:
# Picked the 5 biggest suppliers from import_by_country_no_israel_spikes and added up their shares in each year.
# Why: this sets the top 5 beside the top 5 from import_share_by_year, to see if the spike rows change the picture.
# Found: the shape is the same: between three fifths and two thirds in 2022 to 2024, about half in 2025, up again in 2026.
# Found: only 2022 moves, up a little without the spike rows.
# Decision: the top 5 result does not depend on the spike rows, so report it as it is.
import_share_by_year_no_israel_spikes.loc[import_by_country_no_israel_spikes.head(5).index].sum()

,0
year,
2022,0.661214
2023,0.615987
2024,0.609261
2025,0.501222
2026,0.578343


In [ ]:
# Built monthly_no_estimates: the monthly totals worked out again without the Estimates rows in EU - Imports, with imports,
# exports, the gap and their 12-month averages built as monthly was.
# Why: check 6 flagged these HMRC estimate rows for Analyse. They are in every total, so this shows how far they could move
# the 12-month averages.
# Found: the 55 rows hold £18,609,563 in all, which is 0.0089% of the value in country_clean.
# Found: leaving them out changes no 12-month average, of imports or of the gap, by more than £1,525,933.
# Decision: keep the rows in every table, because the most they move any 12-month average is about £1.5m.
estimates_eu_imports = country_clean[(country_clean['row_type'] == 'estimate') & (country_clean['flow_type'] == 'EU - Imports')]
monthly_no_estimates = country_clean.drop(estimates_eu_imports.index).pivot_table(values='value_gbp', columns='flow_type', aggfunc='sum', index='date')
monthly_no_estimates['imports'] = monthly_no_estimates['EU - Imports'] + monthly_no_estimates['Non EU - Imports']
monthly_no_estimates['exports'] = monthly_no_estimates['EU - Exports'] + monthly_no_estimates['Non EU - Exports']
monthly_no_estimates['gap'] = monthly_no_estimates['exports'] - monthly_no_estimates['imports']
monthly_no_estimates['imports_12m'] = monthly_no_estimates['imports'].rolling(12).mean()
monthly_no_estimates['gap_12m'] = monthly_no_estimates['gap'].rolling(12).mean()
print('Estimates rows in EU - Imports:', len(estimates_eu_imports), 'rows, £{:,}'.format(estimates_eu_imports['value_gbp'].sum()))
print('Share of all value in country_clean: {:.4f}%'.format(estimates_eu_imports['value_gbp'].sum() / country_clean['value_gbp'].sum() * 100))
print('Largest change in a 12-month imports average: £{:,.0f}'.format((monthly['imports_12m'] - monthly_no_estimates['imports_12m']).abs().max()))
print('Largest change in a 12-month gap average: £{:,.0f}'.format((monthly['gap_12m'] - monthly_no_estimates['gap_12m']).abs().max()))

Estimates rows in EU - Imports: 55 rows, £18,609,563
Share of all value in country_clean: 0.0089%
Largest change in a 12-month imports average: £1,525,933
Largest change in a 12-month gap average: £1,525,933


### Step 6: split the totals by medicine

In [ ]:
# Built product_rows from product_clean: only the rows where row_type is product.
# Why: the estimate and below_threshold rows belong to no product group, so they cannot go in a table by product group.
# Found: the product rows hold nearly all of the value in product_clean, so little is left out.
# Decision: use product_rows for every table by product group. Their totals are a little under the full product totals.
product_rows = product_clean[product_clean['row_type'] == 'product']

In [ ]:
product_rows

,hs4,hs6,cn8,value_gbp,net_mass_kg,flow_type,year,month,date,row_type,product_group
218,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,45541,90.0,EU - Exports,2022,January,2022-01-01,product,"Glands, organ extracts, heparin"
219,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,22701,90.0,EU - Exports,2022,February,2022-02-01,product,"Glands, organ extracts, heparin"
220,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,423481,964.0,EU - Exports,2022,March,2022-03-01,product,"Glands, organ extracts, heparin"
221,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,495681,2244.0,EU - Exports,2022,April,2022-04-01,product,"Glands, organ extracts, heparin"
222,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,30500,1.0,EU - Exports,2022,May,2022-05-01,product,"Glands, organ extracts, heparin"
...,...,...,...,...,...,...,...,...,...,...,...
10843,3006 Pharmaceutical preparations and products ...,300693 Placebos and blinded or double-blinded ...,30069300 Placebos and blinded or double-blinde...,1509602,1359.0,Non EU - Imports,2026,March,2026-03-01,product,Other pharmaceutical products
10844,3006 Pharmaceutical preparations and products ...,300693 Placebos and blinded or double-blinded ...,30069300 Placebos and blinded or double-blinde...,809189,1383.0,Non EU - Imports,2026,April,2026-04-01,product,Other pharmaceutical products
10845,3006 Pharmaceutical preparations and products ...,300693 Placebos and blinded or double-blinded ...,30069300 Placebos and blinded or double-blinde...,8564201,2189.0,Non EU - Imports,2026,May,2026-05-01,product,Other pharmaceutical products
10846,3006 Pharmaceutical preparations and products ...,300693 Placebos and blinded or double-blinded ...,30069300 Placebos and blinded or double-blinde...,851389,2137.0,Non EU - Imports,2026,June,2026-06-01,product,Other pharmaceutical products


In [ ]:
product_rows.shape

(10630, 11)

In [ ]:
# Added up value_gbp in product_rows.
# Why: to see how much of the money the product rows hold, compared with all_value.
# Found: a little under all_value, so only a small amount of money sits in the estimate and below_threshold rows.
product_rows['value_gbp'].sum()

np.int64(209603606875)

In [ ]:
# Counted the different flow_type values in product_rows.
# Why: to check that all four flows are still in the product rows after the filter.
# Found: all four are there.
product_rows['flow_type'].nunique()

4

In [ ]:
# Built group_totals from product_rows: one row per product group, one column per flow type, with value_gbp added up
# over the whole period.
# Why: questions 1 and 3 ask whether exports fell and whether imports grew. The product groups show which kinds of medicine
# are behind those changes, and group_totals shows how big each group is.
group_totals = product_rows.pivot_table(index='product_group', values='value_gbp', columns='flow_type', aggfunc='sum')

In [ ]:
group_totals

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports
product_group,,,,
"Blood products, antisera, vaccines",8091223678,25020091336,17427296359,10410342150
Bulk mixed medicines,187636919,359445221,702009324,1392559748
Dressings and bandages,1212148151,1106490130,1045281458,656025758
"Glands, organ extracts, heparin",33391123,90918970,66688375,111452787
Other pharmaceutical products,718564515,2434533776,993788055,381069423
Packaged medicines,34223170588,50336686471,35117410124,17485382436


In [ ]:
# Added imports to group_totals: EU imports plus non-EU imports for each product group.
# Why: question 3 is about imports, so each group needs one imports figure, as monthly has.
group_totals['imports'] = group_totals['EU - Imports'] + group_totals['Non EU - Imports']

In [ ]:
# Added exports to group_totals: EU exports plus non-EU exports for each product group, as imports was added above.
# Why: question 1 is about exports, so each group needs one exports figure.
group_totals['exports'] = group_totals['EU - Exports'] + group_totals['Non EU - Exports']

In [ ]:
group_totals

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,exports
product_group,,,,,,
"Blood products, antisera, vaccines",8091223678,25020091336,17427296359,10410342150,35430433486,25518520037
Bulk mixed medicines,187636919,359445221,702009324,1392559748,1752004969,889646243
Dressings and bandages,1212148151,1106490130,1045281458,656025758,1762515888,2257429609
"Glands, organ extracts, heparin",33391123,90918970,66688375,111452787,202371757,100079498
Other pharmaceutical products,718564515,2434533776,993788055,381069423,2815603199,1712352570
Packaged medicines,34223170588,50336686471,35117410124,17485382436,67822068907,69340580712


In [ ]:
# Checked that exports plus imports in group_totals equals the value_gbp total of product_rows.
# Why: the pivot and the two new columns should keep every pound, with none lost or counted twice.
# Found: the check returned True.
group_totals['exports'].sum() + group_totals['imports'].sum() == (product_rows['value_gbp'].sum())

np.True_

In [ ]:
# Added exports_share to group_totals: the exports of each group divided by the exports of all groups added together.
# Why: a share says how much of the exports each kind of medicine makes up, which is easier to compare than pounds.
group_totals['exports_share'] = group_totals['exports'] / group_totals['exports'].sum()

In [ ]:
# Added imports_share to group_totals: the imports of each group divided by the imports of all groups added together.
# Why: the same view for imports, to set beside exports_share.
group_totals['imports_share'] = group_totals['imports'] / group_totals['imports'].sum()

In [ ]:
# Showed group_totals with the two share columns.
# Found: packaged medicines are about seven tenths of exports and about three fifths of imports. Blood products are about
# a quarter of exports and about a third of imports. The other groups are each small.
# Decision: describe packaged medicines and blood products first, and mention the other groups only where they explain
# a change.
group_totals

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,exports,exports_share,imports_share
product_group,,,,,,,,
"Blood products, antisera, vaccines",8091223678,25020091336,17427296359,10410342150,35430433486,25518520037,0.255649,0.322726
Bulk mixed medicines,187636919,359445221,702009324,1392559748,1752004969,889646243,0.008913,0.015959
Dressings and bandages,1212148151,1106490130,1045281458,656025758,1762515888,2257429609,0.022615,0.016054
"Glands, organ extracts, heparin",33391123,90918970,66688375,111452787,202371757,100079498,0.001003,0.001843
Other pharmaceutical products,718564515,2434533776,993788055,381069423,2815603199,1712352570,0.017155,0.025647
Packaged medicines,34223170588,50336686471,35117410124,17485382436,67822068907,69340580712,0.694666,0.617772


In [ ]:
# Checked that exports_share adds up to 1 across the product groups.
# Why: shares are parts of one whole, so they add up to 1 if every group is counted once.
# Found: the sum is 1.0.
group_totals['exports_share'].sum()

np.float64(1.0)

In [ ]:
# Checked that imports_share adds up to 1 across the product groups, as exports_share did above.
# Found: the sum is 1.0.
group_totals['imports_share'].sum()

np.float64(1.0)

In [ ]:
# Built product_exports from product_rows: only the two export flows.
# Why: the export table by group and year is built from these rows, as export_partners was built for the countries.
product_exports = product_rows[product_rows['flow_type'].isin(['EU - Exports', 'Non EU - Exports'])]

In [ ]:
product_exports.head(5)

,hs4,hs6,cn8,value_gbp,net_mass_kg,flow_type,year,month,date,row_type,product_group
218,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,45541,90.0,EU - Exports,2022,January,2022-01-01,product,"Glands, organ extracts, heparin"
219,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,22701,90.0,EU - Exports,2022,February,2022-02-01,product,"Glands, organ extracts, heparin"
220,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,423481,964.0,EU - Exports,2022,March,2022-03-01,product,"Glands, organ extracts, heparin"
221,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,495681,2244.0,EU - Exports,2022,April,2022-04-01,product,"Glands, organ extracts, heparin"
222,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,30500,1.0,EU - Exports,2022,May,2022-05-01,product,"Glands, organ extracts, heparin"


In [ ]:
# Built export_by_group_year from product_exports: product groups down the side, years across, value_gbp added up in
# each cell.
# Why: question 1 asks whether exports are higher or lower than before, and one column per year shows which kinds of
# medicine moved.
# Found: exports of packaged medicines fall each year from 2022 to 2025, while exports of blood products rise each year
# over the same years.
# Decision: compare 2022 to 2025 as full years. 2026 holds January to July only, so show it as a part year beside them.
export_by_group_year = product_exports.pivot_table(index='product_group', columns='year', values='value_gbp', aggfunc='sum')

In [ ]:
export_by_group_year.shape

(6, 5)

In [ ]:
export_by_group_year

year,2022,2023,2024,2025,2026
product_group,,,,,
"Blood products, antisera, vaccines",4779142601,5555930772,5586971199,5908288586,3688186879
Bulk mixed medicines,154374499,201123318,214465972,202560918,117121536
Dressings and bandages,525703655,499054142,512285958,473823242,246562612
"Glands, organ extracts, heparin",22695195,24165498,21434894,19053600,12730311
Other pharmaceutical products,291989992,387218241,364330069,438292759,230521509
Packaged medicines,16818164432,15457185260,14983940274,13938891578,8142399168


In [ ]:
# Built product_imports from product_rows: only the two import flows, as product_exports was built for exports.
# Why: the import table by group and year is built from these rows.
product_imports = product_rows[product_rows['flow_type'].isin(['EU - Imports', 'Non EU - Imports'])]

In [ ]:
# Built import_by_group_year from product_imports: product groups down the side, years across, value_gbp added up in
# each cell.
# Why: question 3 asks whether the UK leans more on imports, and one column per year shows which kinds of medicine it buys.
# Found: imports of packaged medicines dip in 2023, then rise in 2024 and 2025. Imports of blood products are lower in 2025
# than in 2022.
# Found: imports of bulk mixed medicines fall sharply from 2022 to 2023 and keep falling.
# Decision: look at bulk mixed medicines imports month by month in the cells below.
import_by_group_year = product_imports.pivot_table(index='product_group', columns='year', values='value_gbp', aggfunc='sum')

In [ ]:
import_by_group_year

year,2022,2023,2024,2025,2026
product_group,,,,,
"Blood products, antisera, vaccines",8372774364,7463620520,7992130222,7142232264,4459676116
Bulk mixed medicines,1063266184,303464245,153396794,140903920,90973826
Dressings and bandages,444271365,367121937,372145781,363660945,215315860
"Glands, organ extracts, heparin",41492986,47158620,38856219,45977059,28886873
Other pharmaceutical products,558901090,572223771,618634606,657769496,408074236
Packaged medicines,16186848402,13232732376,14270621194,15572136550,8559730385


In [ ]:
group_totals.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,exports,exports_share,imports_share
product_group,,,,,,,,
"Blood products, antisera, vaccines",8091223678,25020091336,17427296359,10410342150,35430433486,25518520037,0.255649,0.322726
Bulk mixed medicines,187636919,359445221,702009324,1392559748,1752004969,889646243,0.008913,0.015959
Dressings and bandages,1212148151,1106490130,1045281458,656025758,1762515888,2257429609,0.022615,0.016054
"Glands, organ extracts, heparin",33391123,90918970,66688375,111452787,202371757,100079498,0.001003,0.001843
Other pharmaceutical products,718564515,2434533776,993788055,381069423,2815603199,1712352570,0.017155,0.025647


In [ ]:
# Checked that the two group-by-year tables add up to the same money as group_totals.
# Why: the by-year tables were built separately from group_totals, so a match shows no rows were lost or counted twice.
# Found: the check returned True.
import_by_group_year.sum().sum() + export_by_group_year.sum().sum() == (group_totals['imports'].sum() + group_totals['exports'].sum())

np.True_

In [ ]:
# Worked out gap_by_group_year: export_by_group_year minus import_by_group_year, for each product group and year.
# Below zero, the UK bought more than it sold.
# Why: question 3 is about the gap between exports and imports, and this shows which kinds of medicine it comes from.
# Found: packaged medicines are in surplus from 2022 to 2024 and in deficit in 2025 and 2026. Blood products are in deficit
# in every year, and the deficit is smaller in 2025 than in 2022.
# Found: dressings and bandages are in surplus in every year. Bulk mixed medicines have a large deficit in 2022 for such a
# small group, and it shrinks to almost nothing by 2023.
# Decision: report what moved and by how much, not why. Customs data shows which groups moved, not the cause.
gap_by_group_year = export_by_group_year - import_by_group_year

In [ ]:
gap_by_group_year

year,2022,2023,2024,2025,2026
product_group,,,,,
"Blood products, antisera, vaccines",-3593631763,-1907689748,-2405159023,-1233943678,-771489237
Bulk mixed medicines,-908891685,-102340927,61069178,61656998,26147710
Dressings and bandages,81432290,131932205,140140177,110162297,31246752
"Glands, organ extracts, heparin",-18797791,-22993122,-17421325,-26923459,-16156562
Other pharmaceutical products,-266911098,-185005530,-254304537,-219476737,-177552727
Packaged medicines,631316030,2224452884,713319080,-1633244972,-417331217


In [ ]:
# Checked that the total of gap_by_group_year equals the total exports minus the total imports in group_totals.
# Why: table minus table lines up rows and columns by their labels, so a group or year missing from one table would show up
# here as a mismatch.
# Found: the check returned True.
gap_by_group_year.sum().sum() == (group_totals['exports'].sum() - group_totals['imports'].sum())

np.True_

In [ ]:
product_rows

,hs4,hs6,cn8,value_gbp,net_mass_kg,flow_type,year,month,date,row_type,product_group
218,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,45541,90.0,EU - Exports,2022,January,2022-01-01,product,"Glands, organ extracts, heparin"
219,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,22701,90.0,EU - Exports,2022,February,2022-02-01,product,"Glands, organ extracts, heparin"
220,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,423481,964.0,EU - Exports,2022,March,2022-03-01,product,"Glands, organ extracts, heparin"
221,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,495681,2244.0,EU - Exports,2022,April,2022-04-01,product,"Glands, organ extracts, heparin"
222,3001 Dried glands and other organs for organo-...,300120 Extracts of glands or other organs or o...,30012010 Extracts of glands or other organs or...,30500,1.0,EU - Exports,2022,May,2022-05-01,product,"Glands, organ extracts, heparin"
...,...,...,...,...,...,...,...,...,...,...,...
10843,3006 Pharmaceutical preparations and products ...,300693 Placebos and blinded or double-blinded ...,30069300 Placebos and blinded or double-blinde...,1509602,1359.0,Non EU - Imports,2026,March,2026-03-01,product,Other pharmaceutical products
10844,3006 Pharmaceutical preparations and products ...,300693 Placebos and blinded or double-blinded ...,30069300 Placebos and blinded or double-blinde...,809189,1383.0,Non EU - Imports,2026,April,2026-04-01,product,Other pharmaceutical products
10845,3006 Pharmaceutical preparations and products ...,300693 Placebos and blinded or double-blinded ...,30069300 Placebos and blinded or double-blinde...,8564201,2189.0,Non EU - Imports,2026,May,2026-05-01,product,Other pharmaceutical products
10846,3006 Pharmaceutical preparations and products ...,300693 Placebos and blinded or double-blinded ...,30069300 Placebos and blinded or double-blinde...,851389,2137.0,Non EU - Imports,2026,June,2026-06-01,product,Other pharmaceutical products


In [ ]:
# Built bulk_non_eu_imports from product_rows: only the Bulk mixed medicines rows in the Non EU - Imports flow.
# Why: imports of this group fall sharply after 2022, and most of them are Non EU - Imports, the flow the Israel spike rows
# sit in. This looks at that group and flow on their own.
bulk_non_eu_imports = product_rows[(product_rows['product_group'] == 'Bulk mixed medicines') & (product_rows['flow_type'] == 'Non EU - Imports')]

In [ ]:
bulk_non_eu_imports

,hs4,hs6,cn8,value_gbp,net_mass_kg,flow_type,year,month,date,row_type,product_group
4119,3003 Medicaments consisting of two or more con...,300310 Medicaments containing penicillins or d...,30031000 Medicaments containing penicillins or...,8726,111.0,Non EU - Imports,2022,January,2022-01-01,product,Bulk mixed medicines
4120,3003 Medicaments consisting of two or more con...,300310 Medicaments containing penicillins or d...,30031000 Medicaments containing penicillins or...,12204,61.0,Non EU - Imports,2022,April,2022-04-01,product,Bulk mixed medicines
4121,3003 Medicaments consisting of two or more con...,300310 Medicaments containing penicillins or d...,30031000 Medicaments containing penicillins or...,3515,31.0,Non EU - Imports,2022,July,2022-07-01,product,Bulk mixed medicines
4122,3003 Medicaments consisting of two or more con...,300310 Medicaments containing penicillins or d...,30031000 Medicaments containing penicillins or...,48745,1.0,Non EU - Imports,2022,August,2022-08-01,product,Bulk mixed medicines
4123,3003 Medicaments consisting of two or more con...,300310 Medicaments containing penicillins or d...,30031000 Medicaments containing penicillins or...,1338,14.0,Non EU - Imports,2022,September,2022-09-01,product,Bulk mixed medicines
...,...,...,...,...,...,...,...,...,...,...,...
5088,3003 Medicaments consisting of two or more con...,300390 Medicaments consisting of two or more c...,30039000 Medicaments consisting of two or more...,6004361,54924.0,Non EU - Imports,2026,March,2026-03-01,product,Bulk mixed medicines
5089,3003 Medicaments consisting of two or more con...,300390 Medicaments consisting of two or more c...,30039000 Medicaments consisting of two or more...,3594749,43056.0,Non EU - Imports,2026,April,2026-04-01,product,Bulk mixed medicines
5090,3003 Medicaments consisting of two or more con...,300390 Medicaments consisting of two or more c...,30039000 Medicaments consisting of two or more...,1256212,47187.0,Non EU - Imports,2026,May,2026-05-01,product,Bulk mixed medicines
5091,3003 Medicaments consisting of two or more con...,300390 Medicaments consisting of two or more c...,30039000 Medicaments consisting of two or more...,3746632,29344.0,Non EU - Imports,2026,June,2026-06-01,product,Bulk mixed medicines


In [ ]:
# Built bulk_import_by_month from bulk_non_eu_imports: one row per date, with value_gbp added up for that month.
# Why: with one value per month, the months can be ranked to see whether a few of them hold most of the money.
bulk_import_by_month = bulk_non_eu_imports.pivot_table(index='date', values='value_gbp', aggfunc='sum')

In [ ]:
bulk_import_by_month.head(5)

,value_gbp
date,
2022-01-01,3005818
2022-02-01,5399787
2022-03-01,88768938
2022-04-01,278358859
2022-05-01,81742083


In [ ]:
# Checked that the total of bulk_import_by_month equals the Bulk mixed medicines figure in the Non EU - Imports column
# of group_totals.
# Why: the monthly table came from a filtered copy, so a match shows the filter and the pivot kept every row they should.
# Found: the check returned True.
bulk_import_by_month['value_gbp'].sum() == (group_totals.loc['Bulk mixed medicines', 'Non EU - Imports'])

np.True_

In [ ]:
# Sorted bulk_import_by_month from the biggest month down, as bulk_months_ranked.
# Why: the biggest months show whether the money sits in a few months or is spread across them.
bulk_months_ranked = bulk_import_by_month.sort_values('value_gbp', ascending=False)


In [ ]:
# Showed the biggest months in bulk_months_ranked.
# Why: to see how far the top months stand above the rest.
# Found: several months between March 2022 and April 2023 sit far above all the others. The next month down is less than
# half the size of the smallest of them.
bulk_months_ranked.head(10)

,value_gbp
date,
2022-04-01,278358859
2022-08-01,220073964
2022-06-01,107667985
2023-04-01,99660330
2022-11-01,92493427
2022-09-01,90772101
2022-03-01,88768938
2022-05-01,81742083
2024-02-01,33923444


In [ ]:
israel_spikes

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
9174,NON-EU,Middle East and N Africa,Israel,77505397,4322.0,Non EU - Imports,2022,March,2022-03-01,partner
9175,NON-EU,Middle East and N Africa,Israel,278201892,33131.0,Non EU - Imports,2022,April,2022-04-01,partner
9176,NON-EU,Middle East and N Africa,Israel,78926845,19202.0,Non EU - Imports,2022,May,2022-05-01,partner
9177,NON-EU,Middle East and N Africa,Israel,105149404,6619.0,Non EU - Imports,2022,June,2022-06-01,partner
9179,NON-EU,Middle East and N Africa,Israel,219536263,2840.0,Non EU - Imports,2022,August,2022-08-01,partner
9180,NON-EU,Middle East and N Africa,Israel,88455744,20028.0,Non EU - Imports,2022,September,2022-09-01,partner
9182,NON-EU,Middle East and N Africa,Israel,90019127,2578.0,Non EU - Imports,2022,November,2022-11-01,partner
9187,NON-EU,Middle East and N Africa,Israel,85719239,1234.0,Non EU - Imports,2023,April,2023-04-01,partner


In [ ]:
# Checked whether the biggest months in bulk_months_ranked, taking as many as israel_spikes has rows, all appear in the date
# column of israel_spikes.
# Why: the spike months were found in the country table. The product table comes from the same HMRC data but is a separate
# table, so if the same months stand out, the two tables point to the same trade.
# Found: the check returned True. The months that stand out in bulk mixed medicines imports are the Israel spike months.
# Decision: keep these months in the group tables, as in the country tables. In the write-up, say that the large 2022 figure
# for bulk mixed medicines imports comes from these months.
bulk_months_ranked.head(8).index.isin(israel_spikes['date']).all()

np.True_

### Step 7: work out value per kg and what moved it

In [ ]:
# Built country_with_weight from country_clean: only the rows where net_mass_kg is above zero.
# Why: value per kg divides value by weight, and a blank or zero weight gives no usable figure. Estimate rows have no weight,
# so they drop out here too.
country_with_weight = country_clean[country_clean['net_mass_kg'] > 0]

In [ ]:
country_clean.shape

(12775, 10)

In [ ]:
country_with_weight

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
0,EU,European Union,Austria,6814787,48756.0,EU - Exports,2022,January,2022-01-01,partner
1,EU,European Union,Austria,11891585,74395.0,EU - Exports,2022,February,2022-02-01,partner
2,EU,European Union,Austria,7369406,65655.0,EU - Exports,2022,March,2022-03-01,partner
3,EU,European Union,Austria,7723548,64349.0,EU - Exports,2022,April,2022-04-01,partner
4,EU,European Union,Austria,14518099,17780.0,EU - Exports,2022,May,2022-05-01,partner
...,...,...,...,...,...,...,...,...,...,...
12771,NON-EU,Western Europe exc EU,Turkey,1490026,129853.0,Non EU - Imports,2026,March,2026-03-01,partner
12772,NON-EU,Western Europe exc EU,Turkey,1840808,129637.0,Non EU - Imports,2026,April,2026-04-01,partner
12773,NON-EU,Western Europe exc EU,Turkey,1167245,61088.0,Non EU - Imports,2026,May,2026-05-01,partner
12774,NON-EU,Western Europe exc EU,Turkey,1371726,111984.0,Non EU - Imports,2026,June,2026-06-01,partner


In [ ]:
# Worked out the value_gbp carried by the rows left out of country_with_weight: the country_clean total minus the
# country_with_weight total.
# Why: value per kg leaves these rows out, so this shows how much money that is.
country_clean['value_gbp'].sum() - country_with_weight['value_gbp'].sum()

np.int64(41061896)

In [ ]:
# Divided the money left out by the value_gbp total of country_clean, to get it as a share.
# Why: a share says whether leaving the rows out could change the picture.
# Found: the share is tiny, well under a tenth of one percent.
# Decision: leave the rows without a usable weight out of value per kg. The value tables keep them.
(country_clean['value_gbp'].sum() - country_with_weight['value_gbp'].sum()) / country_clean['value_gbp'].sum()

np.float64(0.0001957951100552088)

In [ ]:
# Built value_by_month from country_with_weight: one row per date, one column per flow type, with value_gbp added up.
# Why: value per kg needs the value and the weight of the same rows, so the monthly value is rebuilt from
# country_with_weight instead of taken from monthly.
value_by_month = country_with_weight.pivot_table(index='date', values='value_gbp', aggfunc='sum', columns='flow_type')

In [ ]:
value_by_month

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports
date,,,,
2022-01-01,472109641,1154639185,700449752,514876233
2022-02-01,916686393,1301958533,808311228,621125576
2022-03-01,1085905563,1600528807,944109239,805618534
2022-04-01,728017451,1637587182,691283385,795310013
2022-05-01,748623268,1988673118,1139327895,640239212
2022-06-01,921091886,1459686716,1013849341,643884445
2022-07-01,998558594,1406419197,1042347057,639969490
2022-08-01,1135006032,1757632396,985261958,569166436
2022-09-01,1070811677,1584242818,963078270,623025844


In [ ]:
# Built kg_by_month from country_with_weight in the same layout as value_by_month, with net_mass_kg added up.
# Why: value per kg is value divided by kilos, so the two tables need the same rows and the same layout.
kg_by_month = country_with_weight.pivot_table(index='date', values='net_mass_kg', aggfunc='sum', columns='flow_type')

In [ ]:
kg_by_month

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports
date,,,,
2022-01-01,6644005.0,22063291.0,5724112.0,6894921.0
2022-02-01,11341579.0,18496317.0,8876023.0,5391007.0
2022-03-01,11974515.0,15180832.0,8477944.0,6741414.0
2022-04-01,9654169.0,17087622.0,7303538.0,4417257.0
2022-05-01,10070500.0,19183085.0,6022154.0,5113774.0
2022-06-01,10112917.0,18226183.0,7152229.0,4989089.0
2022-07-01,10767596.0,19205237.0,7574976.0,4939796.0
2022-08-01,9280705.0,16604588.0,7170154.0,4859532.0
2022-09-01,9379818.0,13658951.0,6720644.0,6177125.0


In [ ]:
# Checked that the total of value_by_month equals the value_gbp total of country_with_weight.
# Why: the pivot should keep every pound of the rows it was built from.
# Found: the check returned True.
value_by_month.sum().sum() == (country_with_weight['value_gbp'].sum())

np.True_

In [ ]:
# Checked that the total of kg_by_month equals the net_mass_kg total of country_with_weight, as for value above.
# Found: the check returned True.
kg_by_month.sum().sum() == (country_with_weight['net_mass_kg'].sum())

np.True_

In [ ]:
# Added imports to value_by_month: EU imports plus non-EU imports for each month.
# Why: value per kg is worked out for all exports and for all imports, so each month needs one figure for each,
# as monthly has.
value_by_month['imports'] = (value_by_month['EU - Imports']) + (value_by_month['Non EU - Imports'])

In [ ]:
# Added exports to value_by_month: EU exports plus non-EU exports for each month, as imports was added above.
value_by_month['exports'] = (value_by_month['EU - Exports']) + (value_by_month['Non EU - Exports'])

In [ ]:
# Added imports to kg_by_month: EU imports plus non-EU imports for each month, built the same way as in value_by_month.
kg_by_month['imports'] = (kg_by_month['EU - Imports']) + (kg_by_month['Non EU - Imports'])

In [ ]:
# Added exports to kg_by_month: EU exports plus non-EU exports for each month, built the same way as in value_by_month.
kg_by_month['exports'] = (kg_by_month['EU - Exports']) + (kg_by_month['Non EU - Exports'])

In [ ]:
country_with_weight.head(5)

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
0,EU,European Union,Austria,6814787,48756.0,EU - Exports,2022,January,2022-01-01,partner
1,EU,European Union,Austria,11891585,74395.0,EU - Exports,2022,February,2022-02-01,partner
2,EU,European Union,Austria,7369406,65655.0,EU - Exports,2022,March,2022-03-01,partner
3,EU,European Union,Austria,7723548,64349.0,EU - Exports,2022,April,2022-04-01,partner
4,EU,European Union,Austria,14518099,17780.0,EU - Exports,2022,May,2022-05-01,partner


In [ ]:
# Checked that imports plus exports in value_by_month equals the value_gbp total of country_with_weight.
# Why: the two new columns should cover every flow, with nothing counted twice.
# Found: the check returned True.
country_with_weight['value_gbp'].sum() == (value_by_month['imports'].sum() + value_by_month['exports'].sum())

np.True_

In [ ]:
# Checked the same for kilos: imports plus exports in kg_by_month equals the net_mass_kg total of country_with_weight.
# Found: the check returned True.
country_with_weight['net_mass_kg'].sum() == (kg_by_month['imports'].sum() + kg_by_month['exports'].sum())

np.True_

In [ ]:
value_by_month.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,exports
date,,,,,,
2022-01-01,472109641,1154639185,700449752,514876233,1669515418,1172559393
2022-02-01,916686393,1301958533,808311228,621125576,1923084109,1724997621
2022-03-01,1085905563,1600528807,944109239,805618534,2406147341,2030014802
2022-04-01,728017451,1637587182,691283385,795310013,2432897195,1419300836
2022-05-01,748623268,1988673118,1139327895,640239212,2628912330,1887951163


In [ ]:
# Added exports_12m to value_by_month: exports added up over each month and the 11 months before it, a 12-month total.
# Why: value per kg needs value and kilos over the same window, and single months swing too much. A 12-month total covers
# every season once and reads as the year up to that month.
# Found: the first 11 months are NaN, because they do not have 12 months to use.
value_by_month['exports_12m'] = value_by_month['exports'].rolling(12).sum()

In [ ]:
# Added imports_12m to value_by_month: imports added up over each month and the 11 months before it, as for exports_12m.
value_by_month['imports_12m'] = value_by_month['imports'].rolling(12).sum()

In [ ]:
# Added exports_12m to kg_by_month: exports in kilos added up over each month and the 11 months before it.
# Why: the kilos need the same 12-month window as the value, so the two can be divided.
kg_by_month['exports_12m'] = kg_by_month['exports'].rolling(12).sum()

In [ ]:
# Added imports_12m to kg_by_month: imports in kilos added up over each month and the 11 months before it.
kg_by_month['imports_12m'] = kg_by_month['imports'].rolling(12).sum()

In [ ]:
kg_by_month.head(15)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,exports,exports_12m,imports_12m
date,,,,,,,,
2022-01-01,6644005.0,22063291.0,5724112.0,6894921.0,28958212.0,12368117.0,NaN,NaN
2022-02-01,11341579.0,18496317.0,8876023.0,5391007.0,23887324.0,20217602.0,NaN,NaN
2022-03-01,11974515.0,15180832.0,8477944.0,6741414.0,21922246.0,20452459.0,NaN,NaN
2022-04-01,9654169.0,17087622.0,7303538.0,4417257.0,21504879.0,16957707.0,NaN,NaN
2022-05-01,10070500.0,19183085.0,6022154.0,5113774.0,24296859.0,16092654.0,NaN,NaN
2022-06-01,10112917.0,18226183.0,7152229.0,4989089.0,23215272.0,17265146.0,NaN,NaN
2022-07-01,10767596.0,19205237.0,7574976.0,4939796.0,24145033.0,18342572.0,NaN,NaN
2022-08-01,9280705.0,16604588.0,7170154.0,4859532.0,21464120.0,16450859.0,NaN,NaN
2022-09-01,9379818.0,13658951.0,6720644.0,6177125.0,19836076.0,16100462.0,NaN,NaN


In [ ]:
value_by_month.head(15)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,exports,exports_12m,imports_12m
date,,,,,,,,
2022-01-01,472109641,1154639185,700449752,514876233,1669515418,1172559393,NaN,NaN
2022-02-01,916686393,1301958533,808311228,621125576,1923084109,1724997621,NaN,NaN
2022-03-01,1085905563,1600528807,944109239,805618534,2406147341,2030014802,NaN,NaN
2022-04-01,728017451,1637587182,691283385,795310013,2432897195,1419300836,NaN,NaN
2022-05-01,748623268,1988673118,1139327895,640239212,2628912330,1887951163,NaN,NaN
2022-06-01,921091886,1459686716,1013849341,643884445,2103571161,1934941227,NaN,NaN
2022-07-01,998558594,1406419197,1042347057,639969490,2046388687,2040905651,NaN,NaN
2022-08-01,1135006032,1757632396,985261958,569166436,2326798832,2120267990,NaN,NaN
2022-09-01,1070811677,1584242818,963078270,623025844,2207268662,2033889947,NaN,NaN


In [ ]:
# Added exports_per_kg to value_by_month: exports_12m in value_by_month divided by exports_12m in kg_by_month.
# Why: this is the supporting check on question 1. When export value moves, value per kg helps show whether the kilos moved
# or the pounds earned for each kilo.
# Decision: call it value per kg, not price. It also moves when the mix of medicines changes, for example when more of the
# kilos are the dearer kinds.
value_by_month['exports_per_kg'] = (value_by_month['exports_12m']) / (kg_by_month['exports_12m'])

In [ ]:
# Added imports_per_kg to value_by_month: imports_12m in value_by_month divided by imports_12m in kg_by_month.
# Why: the same view for imports, to set beside exports_per_kg.
value_by_month['imports_per_kg'] = (value_by_month['imports_12m']) / (kg_by_month['imports_12m'])

In [ ]:
value_by_month.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,exports,exports_12m,imports_12m,exports_per_kg,imports_per_kg
date,,,,,,,,,,
2022-01-01,472109641,1154639185,700449752,514876233,1669515418,1172559393,NaN,NaN,NaN,NaN
2022-02-01,916686393,1301958533,808311228,621125576,1923084109,1724997621,NaN,NaN,NaN,NaN
2022-03-01,1085905563,1600528807,944109239,805618534,2406147341,2030014802,NaN,NaN,NaN,NaN
2022-04-01,728017451,1637587182,691283385,795310013,2432897195,1419300836,NaN,NaN,NaN,NaN
2022-05-01,748623268,1988673118,1139327895,640239212,2628912330,1887951163,NaN,NaN,NaN,NaN


In [ ]:
# Picked the December rows and the latest month from value_by_month, with imports_per_kg and exports_per_kg.
# Why: the December rows are full years, so they can be compared year by year.
# Found: exports are worth more per kilo than imports in every row picked. Exports rise from 2022 to 2023 and ease in every
# row after that. Imports fall in 2023, rise in 2024 and then ease slightly.
# Decision: the import rows for 2022 and 2023 include the Israel spike rows, which carry a lot of money for very little
# weight. Work imports out again without them, further down.
value_by_month.loc[['2022-12-01', '2023-12-01', '2024-12-01', '2025-12-01', '2026-07-01'], ['imports_per_kg', 'exports_per_kg']]

flow_type,imports_per_kg,exports_per_kg
date,,
2022-12-01,97.621386,106.730939
2023-12-01,89.695775,113.739391
2024-12-01,98.185021,112.255475
2025-12-01,96.856612,110.166171
2026-07-01,96.521074,106.145091


In [ ]:
value_by_month.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,exports,exports_12m,imports_12m,exports_per_kg,imports_per_kg
date,,,,,,,,,,
2022-01-01,472109641,1154639185,700449752,514876233,1669515418,1172559393,NaN,NaN,NaN,NaN
2022-02-01,916686393,1301958533,808311228,621125576,1923084109,1724997621,NaN,NaN,NaN,NaN
2022-03-01,1085905563,1600528807,944109239,805618534,2406147341,2030014802,NaN,NaN,NaN,NaN
2022-04-01,728017451,1637587182,691283385,795310013,2432897195,1419300836,NaN,NaN,NaN,NaN
2022-05-01,748623268,1988673118,1139327895,640239212,2628912330,1887951163,NaN,NaN,NaN,NaN


In [ ]:
monthly.head()

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,exports,imports,gap,exports_12m,imports_12m,gap_12m,exports_yoy,imports_yoy,gap_yoy_gbp
date,,,,,,,,,,,,,
2022-01-01,472109643,1154639185,700463239,514876233,1172572882,1669515418,-496942536,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-01,918037338,1301958533,808339491,621125576,1726376829,1923084109,-196707280,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-01,1086990323,1600528807,944113574,805618534,2031103897,2406147341,-375043444,NaN,NaN,NaN,NaN,NaN,NaN
2022-04-01,729387199,1637596367,691283385,795310013,1420670584,2432906380,-1012235796,NaN,NaN,NaN,NaN,NaN,NaN
2022-05-01,748633217,1988675442,1139337732,640250255,1887970949,2628925697,-740954748,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Added six year-on-year columns to value_by_month: for exports and for imports, the change in value, in kilos and in
# value per kg, each against the same month a year earlier.
# Why: the three are linked. One plus the change in value equals one plus the change in kilos times one plus the change in
# value per kg, so the split shows whether a move in value came from the kilos or from the value per kg.
# Found: the first rows are NaN, as for exports_yoy in monthly, because each row needs a 12-month total and the one 12 rows
# before it.
value_by_month['exports_value_yoy'] = value_by_month['exports_12m'].pct_change(12)
value_by_month['exports_kg_yoy'] = kg_by_month['exports_12m'].pct_change(12)
value_by_month['exports_per_kg_yoy'] = value_by_month['exports_per_kg'].pct_change(12)
value_by_month['imports_value_yoy'] = value_by_month['imports_12m'].pct_change(12)
value_by_month['imports_kg_yoy'] = kg_by_month['imports_12m'].pct_change(12)
value_by_month['imports_per_kg_yoy'] = value_by_month['imports_per_kg'].pct_change(12)

In [ ]:
value_by_month.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,exports,exports_12m,imports_12m,exports_per_kg,imports_per_kg,exports_value_yoy,exports_kg_yoy,exports_per_kg_yoy,imports_value_yoy,imports_kg_yoy,imports_per_kg_yoy
date,,,,,,,,,,,,,,,,
2022-01-01,472109641,1154639185,700449752,514876233,1669515418,1172559393,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-01,916686393,1301958533,808311228,621125576,1923084109,1724997621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-01,1085905563,1600528807,944109239,805618534,2406147341,2030014802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-04-01,728017451,1637587182,691283385,795310013,2432897195,1419300836,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-05-01,748623268,1988673118,1139327895,640239212,2628912330,1887951163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Picked the December rows and the latest month from value_by_month, with the year-on-year changes in value, kilos and
# value per kg for exports and imports.
# Why: the December rows are full years, and the latest month shows where the series ends.
# Found: export value falls in every row picked. The 2023 fall is in kilos, partly offset by a higher value per kg. In the
# latest 12 months the fall is in value per kg, and the kilos rise slightly.
# Found: import value falls in 2023, rises in 2024 and 2025, and is about flat in the latest 12 months. The 2023 fall is in
# both kilos and value per kg, the 2024 rise is in value per kg, and the 2025 rise is in kilos.
# Decision: report the latest export fall as a fall in value per kg, not in weight. The import changes for 2023 and 2024
# include the spike rows, so they are worked out again without them further down.
value_by_month.loc[['2023-12-01', '2024-12-01', '2025-12-01', '2026-07-01'], ['exports_value_yoy', 'exports_kg_yoy', 'exports_per_kg_yoy', 'imports_value_yoy', 'imports_kg_yoy', 'imports_per_kg_yoy']]

flow_type,exports_value_yoy,exports_kg_yoy,exports_per_kg_yoy,imports_value_yoy,imports_kg_yoy,imports_per_kg_yoy
date,,,,,,
2023-12-01,-0.020757,-0.081097,0.065665,-0.175230,-0.102352,-0.081187
2024-12-01,-0.019860,-0.006904,-0.013047,0.066553,-0.025663,0.094645
2025-12-01,-0.032517,-0.014168,-0.018612,0.019817,0.033804,-0.013530
2026-07-01,-0.078684,0.018449,-0.095373,-0.008459,0.003244,-0.011665


In [ ]:
israel_spikes

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
9174,NON-EU,Middle East and N Africa,Israel,77505397,4322.0,Non EU - Imports,2022,March,2022-03-01,partner
9175,NON-EU,Middle East and N Africa,Israel,278201892,33131.0,Non EU - Imports,2022,April,2022-04-01,partner
9176,NON-EU,Middle East and N Africa,Israel,78926845,19202.0,Non EU - Imports,2022,May,2022-05-01,partner
9177,NON-EU,Middle East and N Africa,Israel,105149404,6619.0,Non EU - Imports,2022,June,2022-06-01,partner
9179,NON-EU,Middle East and N Africa,Israel,219536263,2840.0,Non EU - Imports,2022,August,2022-08-01,partner
9180,NON-EU,Middle East and N Africa,Israel,88455744,20028.0,Non EU - Imports,2022,September,2022-09-01,partner
9182,NON-EU,Middle East and N Africa,Israel,90019127,2578.0,Non EU - Imports,2022,November,2022-11-01,partner
9187,NON-EU,Middle East and N Africa,Israel,85719239,1234.0,Non EU - Imports,2023,April,2023-04-01,partner


In [ ]:
# Built country_with_weight_no_spikes: country_with_weight with the israel_spikes rows dropped, by their index labels.
# Why: the spike rows carry a lot of money for very little weight, so they pull import value per kg up. This copy shows the
# import figures without them, to set beside the ones with them.
# Decision: keep country_with_weight whole. The name means without the spike rows, not without Israel.
country_with_weight_no_spikes = country_with_weight.drop(israel_spikes.index)

In [ ]:
country_with_weight_no_spikes.shape

(12549, 10)

In [ ]:
country_with_weight_no_spikes.head(5)

,eu_status,continent,country,value_gbp,net_mass_kg,flow_type,year,month,date,row_type
0,EU,European Union,Austria,6814787,48756.0,EU - Exports,2022,January,2022-01-01,partner
1,EU,European Union,Austria,11891585,74395.0,EU - Exports,2022,February,2022-02-01,partner
2,EU,European Union,Austria,7369406,65655.0,EU - Exports,2022,March,2022-03-01,partner
3,EU,European Union,Austria,7723548,64349.0,EU - Exports,2022,April,2022-04-01,partner
4,EU,European Union,Austria,14518099,17780.0,EU - Exports,2022,May,2022-05-01,partner


In [ ]:
# Built value_by_month_no_spikes from country_with_weight_no_spikes, the same way value_by_month was built.
# Why: the import figures need the same monthly table without the spike rows.
value_by_month_no_spikes = country_with_weight_no_spikes.pivot_table(columns='flow_type', index='date', values='value_gbp', aggfunc='sum')

In [ ]:
value_by_month_no_spikes.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports
date,,,,
2022-01-01,472109641,1154639185,700449752,514876233
2022-02-01,916686393,1301958533,808311228,621125576
2022-03-01,1085905563,1600528807,944109239,728113137
2022-04-01,728017451,1637587182,691283385,517108121
2022-05-01,748623268,1988673118,1139327895,561312367


In [ ]:
# Built kg_by_month_no_spikes from country_with_weight_no_spikes, the same way kg_by_month was built.
kg_by_month_no_spikes = country_with_weight_no_spikes.pivot_table(columns='flow_type', index='date', values='net_mass_kg', aggfunc='sum')

In [ ]:
# Added imports to value_by_month_no_spikes: EU imports plus non-EU imports for each month.
# Why: only imports are worked out here. The spike rows are imports, so exports cannot change.
value_by_month_no_spikes['imports'] = (value_by_month_no_spikes['EU - Imports']) + (value_by_month_no_spikes['Non EU - Imports'])

In [ ]:
# Added imports to kg_by_month_no_spikes: EU imports plus non-EU imports for each month, as in value_by_month_no_spikes.
kg_by_month_no_spikes['imports'] = (kg_by_month_no_spikes['EU - Imports']) + (kg_by_month_no_spikes['Non EU - Imports'])

In [ ]:
value_by_month_no_spikes.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports
date,,,,,
2022-01-01,472109641,1154639185,700449752,514876233,1669515418
2022-02-01,916686393,1301958533,808311228,621125576,1923084109
2022-03-01,1085905563,1600528807,944109239,728113137,2328641944
2022-04-01,728017451,1637587182,691283385,517108121,2154695303
2022-05-01,748623268,1988673118,1139327895,561312367,2549985485


In [ ]:
kg_by_month_no_spikes.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports
date,,,,,
2022-01-01,6644005.0,22063291.0,5724112.0,6894921.0,28958212.0
2022-02-01,11341579.0,18496317.0,8876023.0,5391007.0,23887324.0
2022-03-01,11974515.0,15180832.0,8477944.0,6737092.0,21917924.0
2022-04-01,9654169.0,17087622.0,7303538.0,4384126.0,21471748.0
2022-05-01,10070500.0,19183085.0,6022154.0,5094572.0,24277657.0


In [ ]:
# Added imports_12m to value_by_month_no_spikes: imports added up over each month and the 11 months before it.
value_by_month_no_spikes['imports_12m'] = value_by_month_no_spikes['imports'].rolling(12).sum()

In [ ]:
# Added imports_12m to kg_by_month_no_spikes: imports in kilos added up over each month and the 11 months before it.
kg_by_month_no_spikes['imports_12m'] = kg_by_month_no_spikes['imports'].rolling(12).sum()

In [ ]:
# Added imports_per_kg and the three imports year-on-year columns (value, kilos and value per kg) to
# value_by_month_no_spikes, built the same way as in value_by_month.
# Why: these are the import figures behind question 3 without the spike rows, to set beside the ones with them.
value_by_month_no_spikes['imports_per_kg'] = (value_by_month_no_spikes['imports_12m']) / (kg_by_month_no_spikes['imports_12m'])
value_by_month_no_spikes['imports_value_yoy'] = value_by_month_no_spikes['imports_12m'].pct_change(12)
value_by_month_no_spikes['imports_kg_yoy'] = kg_by_month_no_spikes['imports_12m'].pct_change(12)
value_by_month_no_spikes['imports_per_kg_yoy'] = value_by_month_no_spikes['imports_per_kg'].pct_change(12)

In [ ]:
value_by_month_no_spikes.head(5)

flow_type,EU - Exports,EU - Imports,Non EU - Exports,Non EU - Imports,imports,imports_12m,imports_per_kg,imports_value_yoy,imports_kg_yoy,imports_per_kg_yoy
date,,,,,,,,,,
2022-01-01,472109641,1154639185,700449752,514876233,1669515418,NaN,NaN,NaN,NaN,NaN
2022-02-01,916686393,1301958533,808311228,621125576,1923084109,NaN,NaN,NaN,NaN,NaN
2022-03-01,1085905563,1600528807,944109239,728113137,2328641944,NaN,NaN,NaN,NaN,NaN
2022-04-01,728017451,1637587182,691283385,517108121,2154695303,NaN,NaN,NaN,NaN,NaN
2022-05-01,748623268,1988673118,1139327895,561312367,2549985485,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Picked the December rows and the latest month from value_by_month_no_spikes, with imports_per_kg and the three imports
# year-on-year changes.
# Why: these are the same rows picked from value_by_month earlier, so the two tables can be compared line by line.
# Found: the 2023 fall in import value is about a seventh without the spike rows and about a sixth with them. The change in
# kilos is about the same in both, because the spike rows weigh very little, so the difference is in value per kg.
# Found: 2024 differs only a little. 2025 and the latest month are identical, because those years have no spike rows.
# Found: the pattern of the changes is the same with or without the spike rows.
# Decision: keep the spike months in the main figures, because they are in the customs data and no cause can be shown.
# Quote the figure without them beside the 2023 fall in import value.
value_by_month_no_spikes.loc[['2022-12-01', '2023-12-01', '2024-12-01', '2025-12-01', '2026-07-01'], ['imports_per_kg', 'imports_value_yoy', 'imports_kg_yoy', 'imports_per_kg_yoy']]

flow_type,imports_per_kg,imports_value_yoy,imports_kg_yoy,imports_per_kg_yoy
date,,,,
2022-12-01,94.220287,NaN,NaN,NaN
2023-12-01,89.346785,-0.148511,-0.102065,-0.051725
2024-12-01,98.185021,0.070724,-0.025658,0.098921
2025-12-01,96.856612,0.019817,0.033804,-0.013530
2026-07-01,96.521074,-0.008459,0.003244,-0.011665


### Step 8: save the tables for the Share phase

In [ ]:
# Saved the analysis tables as CSV files in the analysis_tables folder inside the Project 3 folder. The folder has to exist
# before this cell runs, because to_csv does not create folders.
# Why: the Share phase reads these files, so the charts use exactly the numbers worked out here.
# Decision: keep the index when saving, because the dates, countries and product groups are held in the index.
monthly.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/monthly.csv')
monthly_no_israel_spikes.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/monthly_no_israel_spikes.csv')
export_by_country_year.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/export_by_country_year.csv')
export_share_by_year.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/export_share_by_year.csv')
import_by_country_year.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/import_by_country_year.csv')
import_share_by_year.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/import_share_by_year.csv')
import_share_by_year_no_israel_spikes.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/import_share_by_year_no_israel_spikes.csv')
group_totals.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/group_totals.csv')
export_by_group_year.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/export_by_group_year.csv')
import_by_group_year.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/import_by_group_year.csv')
gap_by_group_year.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/gap_by_group_year.csv')
value_by_month.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/value_by_month.csv')
value_by_month_no_spikes.to_csv('/content/drive/MyDrive/Project 3/analysis_tables/value_by_month_no_spikes.csv')